# Chapter 3.2 Design

## 3.2 System Design Overview

Following the established benchmarking principles outlined by Gray [9, Ch.1], Luantick extends the Yardstick framework to provide a protocol-agnostic benchmarking solution for Multi-user Virtual Environments (MVEs). Like all comprehensive benchmarks, Luantick defines: (1) a systematic process to conduct benchmark experiments, including monitoring procedures and experimental protocols, (2) configurable workloads that can be applied to diverse MVE systems under test, and (3) standardized metrics for evaluating system performance and behavior.

Luantick's architecture comprises three primary components that work together to address the requirements established in Section 3.1: **server orchestration**, **bot simulation**, and **network analysis**. Each component is designed with modularity, scalability, and automation as core principles, enabling the framework to adapt to different MVE protocols while maintaining measurement consistency and experimental reproducibility.

## 3.2.1 The Benchmarking Process

The benchmarking process in Luantick addresses requirements R1, R3, R4, R6, and R7 through a systematic experimental methodology. Luantick evaluates MVE server performance using parameterized configurations grouped into discrete experiments, enabling systematic performance analysis across different operational conditions (R1).

Unlike protocol-specific benchmarks, Luantick subjects MVE servers to workloads that are abstracted from the underlying communication protocol. The framework generates workloads through two primary mechanisms: procedural virtual world generation and emulated player behaviors. Both mechanisms are fully parameterizable, creating virtually unlimited workload combinations. MVE services can utilize either procedurally generated environments or pre-configured world states, allowing different server implementations to execute identical workloads for comparative analysis.

The workload generated by emulated players is specified through configurable behavioral models that simulate realistic user interactions (R4). These behavioral patterns are protocol-agnostic, focusing on high-level actions such as movement, interaction, and communication rather than protocol-specific message formats. This abstraction ensures that workloads remain independent of the tested MVE server implementation (R1) while covering diverse operational scenarios representative of real-world usage patterns (R7).

Luantick implements comprehensive monitoring of both server-side and client-side resources during experiment execution. The framework captures system-level metrics including CPU utilization, memory consumption, and network throughput, alongside application-level metrics such as tick duration, message transmission rates, and server responsiveness (R4). Upon experiment completion, both raw measurement data and processed analytical results are made available for performance evaluation (R1, R3).

## 3.2.2 The Benchmark Architecture

The Luantick architecture extends Yardstick's three-component model to accommodate protocol diversity in MVE systems. The **Server Orchestration** component manages the deployment and configuration of MVE servers across distributed infrastructure, handling protocol-specific initialization procedures while maintaining a uniform control interface. This component includes the target MVE service, configuration management, virtual world state, and the integrated monitoring collector that tracks server-side performance metrics.

The **Bot Simulation** component implements protocol-agnostic player emulation through adaptive communication layers. Emulated players connect to MVE systems through protocol-specific adapters that translate high-level behavioral commands into appropriate network messages. The behavioral models are specified independently of protocol details, allowing the same user simulation patterns to be applied across different MVE implementations (R7).

The **Network Analysis** component extends traditional monitoring to capture protocol-agnostic communication patterns and performance characteristics. This component monitors both emulated clients and the virtual environment, processes experimental results, and maintains a comprehensive results database for comparative analysis across different MVE systems and protocols.

## 3.2.3 The Benchmark Implementation

The implementation of Luantick addresses requirements R2, R5, and R8 through careful technology selection and architectural decisions. The monitoring infrastructure leverages Prometheus for time-series data collection, chosen for its portability (R5) and minimal performance overhead (R8). The modular design allows monitoring components to be replaced with alternative systems such as Ganglia or Nagios based on deployment requirements.

Luantick has been deployed and validated on the DAS-5 multi-cluster infrastructure [2], which provides representative hardware characteristics found in modern distributed computing environments, including x86-based processors, high-speed memory and storage systems, and both Ethernet and InfiniBand networking capabilities.

The framework collects server performance data through protocol-specific collector components that are designed for minimal computational overhead (R8). These collectors serve dual purposes: capturing tick-related performance data and publishing measurements to the monitoring subsystem. The collector architecture emphasizes low-overhead data collection, with computational processing deferred to the monitoring component to minimize impact on the system under test.

Implementation of collector components follows a protocol-specific approach while maintaining interface consistency. Currently, Luantick includes collectors for multiple MVE systems, implemented as minimal modifications to server source code to reduce implementation complexity (R2). The framework also supports external collection methods, such as Java agents or protocol analyzers, providing flexibility in measurement approaches based on the characteristics of the target MVE system.

This design enables Luantick to maintain the rigor and systematic approach of Yardstick while extending benchmarking capabilities to the diverse landscape of MVE systems and communication protocols.

# Luantick Benchmark Example - Enhanced with Local Logic

This example notebook provides a comprehensive example of how to use the Yardstick benchmark framework to collect performance metrics from Luanti game servers and evaluate their performance under bot load. This version incorporates the local benchmark logic and supports both walkbots and blockbots.

## Running a Luanti Experiment

The cell below shows you how to run a Luanti server performance experiment using the DAS cluster. This deploys a Luanti server on one node and bots (walkbots or blockbots) on other nodes.

In [1]:
import logging
import os
import shutil
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Optional

from yardstick_benchmark.provisioning import Das
from yardstick_benchmark.monitoring import Telegraf
from yardstick_benchmark.games.luanti.server import LuantiServer
from yardstick_benchmark.games.luanti.workload import RustWalkAround, RustBlockBot
import yardstick_benchmark

from time import sleep
import tempfile

# Configure logging for better visibility
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

print("✓ Imports successful")

def check_dependencies():
    """Check if required tools and paths are available."""
    logger.info("Checking dependencies...")
    
    # Check if yardstick_benchmark is properly installed
    try:
        import yardstick_benchmark
        logger.info("✓ yardstick_benchmark module available")
    except ImportError:
        logger.error("✗ yardstick_benchmark module not found")
        raise ImportError("Please ensure the yardstick benchmark framework is properly installed")
    
    # Check if bot components exist
    bot_dir = Path("bot_components/texmodbot")
    if not bot_dir.exists():
        logger.error(f"✗ Rust bot directory not found: {bot_dir}")
        raise FileNotFoundError("Please ensure bot_components/texmodbot exists")
    logger.info(f"✓ Rust bot components found: {bot_dir}")
    
    # Check output directory permissions
    dest = Path(f"/var/scratch/{os.getlogin()}/yardstick/luanti_output")
    try:
        dest.parent.mkdir(parents=True, exist_ok=True)
        logger.info(f"✓ Output directory accessible: {dest}")
    except Exception as e:
        logger.error(f"✗ Cannot access output directory: {e}")
        raise
    
    return dest

# Run dependency check
dest = check_dependencies()
print(f"Dependencies checked successfully. Results will be saved to: {dest}")

# 🎮 LUANTI BENCHMARK CONFIGURATION - UPDATED PROVEN BUILD METHOD
# ================================================================
# This configuration uses our tested and proven build method that works on DAS5

# === MAIN SETTINGS ===
BOTS_PER_NODE = 200          # Number of bots per bot node (tested and working)
BENCHMARK_DURATION = 120    # Benchmark duration in seconds
NUM_NODES = 3          # Start with 2 nodes: 1 server + 1 bot node

# === BOT TYPE SETTINGS ===
BOT_TYPE = "walkbot"        # Use walkbot (proven to work)
MOVEMENT_MODE = "random"    # Random movement (tested)
MOVEMENT_SPEED = 2.0        # Speed in seconds between actions

# === GAME CONFIGURATION ===
GAME_MODE = "minetest_game" # Use standard minetest_game (most reliable)

# === PROVEN BUILD CONFIGURATION ===
# Based on our successful manual testing and documentation
USE_HEADLESS_BUILD = True           # Use headless server build (no client dependencies)
BUILD_WITH_LUAJIT = True            # Build with LuaJIT for better performance
ENABLE_IPV4_ONLY = True             # Use IPv4 only (fixes connection issues)
USE_SYSTEM_LIBS = True              # Use system libraries where available
DISABLE_UNNECESSARY_FEATURES = True  # Disable gettext, client features, etc.

# === NETWORK CONFIGURATION ===
SERVER_PORT = 30000         # Standard Luanti port
SERVER_BIND_ADDRESS = "127.0.0.1"  # IPv4 localhost binding
DISABLE_IPV6 = True         # Explicitly disable IPv6

# === SPAWN AREA POSITIONING ===
SPAWN_X = 0
SPAWN_Y = 9.5
SPAWN_Z = 123
BUILD_NEAR_SPAWN = True     # Position bots near spawn area

# === ADVANCED SETTINGS ===
COLLECT_ALL_NODES = True    # Monitor all nodes (server + all bot nodes)
VERBOSE_PROGRESS = True     # Show detailed progress during benchmark

# === CALCULATED VALUES ===
TOTAL_BOTS = BOTS_PER_NODE * (NUM_NODES - 1)  # Total bots across all bot nodes


for i in range(2, NUM_NODES + 1):
    bot_group = chr(64 + i - 1)  # A, B, C, etc.


expected_runtime = BENCHMARK_DURATION + 300  # 5 minutes overhead for build

2025-08-14 16:28:18 - INFO - Checking dependencies...
2025-08-14 16:28:18 - INFO - ✓ yardstick_benchmark module available
2025-08-14 16:28:18 - INFO - ✓ Rust bot components found: bot_components/texmodbot
2025-08-14 16:28:18 - INFO - ✓ Output directory accessible: /var/scratch/aco237/yardstick/luanti_output


✓ Imports successful
Dependencies checked successfully. Results will be saved to: /var/scratch/aco237/yardstick/luanti_output


## Provision DAS nodes

In [2]:
def provision_nodes_with_validation(num_nodes: int = 2):
    """Provision nodes on the DAS cluster with validation."""
    logger.info(f"Provisioning {num_nodes} nodes on DAS cluster...")
    
    das = Das()
    try:
        nodes = das.provision(num=num_nodes, time_s=5500)
        
        logger.info(f"✓ Successfully provisioned {len(nodes)} nodes:")
        for i, node in enumerate(nodes):
            logger.info(f"  Node {i}: {node.host} (wd: {node.wd})")
        return das, nodes
    except Exception as e:
        logger.error(f"✗ Failed to provision nodes: {e}")
        raise

start_time = datetime.now()


das, nodes = provision_nodes_with_validation(num_nodes=NUM_NODES)

# Remove previous results if they exist
if dest.exists():
    print(f"Removing previous results at {dest}")
    shutil.rmtree(dest)

# Clean any previous data on nodes
yardstick_benchmark.clean(nodes)

2025-08-14 16:28:20 - INFO - Provisioning 3 nodes on DAS cluster...
2025-08-14 16:28:21 - INFO - ✓ Successfully provisioned 3 nodes:
2025-08-14 16:28:21 - INFO -   Node 0: node030 (wd: /local/aco237/yardstick/node030)
2025-08-14 16:28:21 - INFO -   Node 1: node031 (wd: /local/aco237/yardstick/node031)
2025-08-14 16:28:21 - INFO -   Node 2: node032 (wd: /local/aco237/yardstick/node032)


Removing previous results at /var/scratch/aco237/yardstick/luanti_output

PLAY [Clean data from nodes] ***************************************************

TASK [Gathering Facts] *********************************************************
ok: [node030]
ok: [node031]
ok: [node032]

TASK [Remove data from nodes] **************************************************
ok: [node032]
ok: [node031]
changed: [node030]

PLAY RECAP *********************************************************************
node030                    : ok=2    changed=1    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   
node031                    : ok=2    changed=0    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   
node032                    : ok=2    changed=0    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   


## Start Telegraf for metrics

In [ ]:
telegraf = Telegraf(nodes)
    
telegraf.add_input_luanti_metrics(nodes[0])  # Server node
res = telegraf.deploy()
telegraf.start()


In [ ]:
# Minimal Telegraf deploy (system metrics + Luanti tail inputs)
from yardstick_benchmark.monitoring import Telegraf
from pprint import pprint

# Recreate Telegraf if it already existed (idempotent pattern)
try:
    telegraf.stop(); telegraf.cleanup()
except Exception:
    pass

telegraf = Telegraf(nodes)  # all nodes for system metrics
# Register Luanti server node (node 0) BEFORE deploy so Jinja conditional fires
telegraf.add_input_luanti_metrics(nodes[0])

print('Extravars before deploy:')
pprint(telegraf.extravars)
print('\nInventory groups before deploy:')
pprint(list(telegraf.inv.keys()))

print('\nDeploying Telegraf (system + Luanti metrics) ...')
telegraf.deploy()
print('Starting Telegraf ...')
telegraf.start()
print('Telegraf started. You can now (re)deploy/start the Luanti server.')

# Quick remote verification helper (non-fatal): check if luanti tail blocks rendered
import subprocess, textwrap, os
check_cmd = f"grep -n 'luanti_tick_metrics' telegraf.conf || echo 'NO_LUANTI_BLOCKS'"
for n in nodes:
    try:
        out = n.run(f"cd {n.wd}/telegraf; {check_cmd}")
        print(f"[{n.host}] telegraf.conf luanti block grep:\n{out.strip()}\n")
    except Exception as e:
        print(f"[{n.host}] grep failed: {e}")

## Deploy Luanti headless server

In [3]:
luanti_server = LuantiServer(
    nodes[:1], 
    game_mode=GAME_MODE,        # Use our configured game mode
    use_source_build=False,     # Use source build method (for headless)
    enable_luajit=False,        # Enable LuaJIT
    ipv4_only=ENABLE_IPV4_ONLY  # IPv4 only configuration
)

# Debug: Check the script paths
print(f"Deploy script path: {luanti_server.deploy_action.script}")
print(f"Script exists: {luanti_server.deploy_action.script.exists()}")
print(f"Script is file: {luanti_server.deploy_action.script.is_file()}")
print(f"Current working directory: {os.getcwd()}")


try:
    luanti_server.deploy()
    print("✅ Luanti server deployed successfully")
    
    print("🚀 Starting Luanti headless server...")
    luanti_server.start()
    print("✅ Luanti headless server started successfully")
    
    # Give server time to fully initialize
    print("⏳ Allowing server to initialize (10 seconds)...")
    sleep(10)
    
except Exception as e:
    print(f"❌ Error starting Luanti server: {e}")
    print("Server may have failed to bind to the configured address/port.")
    raise

Deploy script path: /var/scratch/aco237/luantick/yardstick_benchmark/games/luanti/server/luanti_deploy.yml
Script exists: True
Script is file: True
Current working directory: /var/scratch/aco237/luantick

PLAY [Deploy Pre-compiled Luanti Server] ***************************************

TASK [Gathering Facts] *********************************************************
ok: [node030]

TASK [Create working directory and subdirectories] *****************************
changed: [node030] => (item=/local/aco237/yardstick/node030/luanti_server-ep3skdub)
changed: [node030] => (item=/local/aco237/yardstick/node030/luanti_server-ep3skdub/lib)
changed: [node030] => (item=/local/aco237/yardstick/node030/luanti_server-ep3skdub/logs)
changed: [node030] => (item=/local/aco237/yardstick/node030/luanti_server-ep3skdub/games)
changed: [node030] => (item=/local/aco237/yardstick/node030/luanti_server-ep3skdub/builtin)
changed: [node030] => (item=/local/aco237/yardstick/node030/luanti_server-ep3skdub/worlds/ben

## 📊 Comprehensive Metrics Verification

This section verifies that both **system metrics** (CPU, RAM) and **application metrics** (ticks per second) are being collected properly. The enhanced collector mod now writes TSV files that Telegraf can monitor.

In [ ]:
# Comprehensive Metrics Collection Verification
import subprocess, time

print("🔧 COMPREHENSIVE METRICS VERIFICATION")
print("=" * 50)

node = nodes[0]

# 1. Verify System Metrics Collection
print("\n🖥️  SYSTEM METRICS VERIFICATION")
print("-" * 30)

# Check CPU usage
cpu_cmd = f"ssh {node.host} 'top -bn1 | grep \"Cpu(s)\" | awk \"{{print \\$2}}\" | cut -d% -f1'"
cpu_result = subprocess.run(cpu_cmd, shell=True, capture_output=True, text=True)
if cpu_result.stdout.strip():
    print(f"✅ CPU Usage: {cpu_result.stdout.strip()}%")
else:
    print("⚠️  CPU metrics: Reading...")

# Check memory usage
mem_cmd = f"ssh {node.host} 'free -h | grep Mem'"
mem_result = subprocess.run(mem_cmd, shell=True, capture_output=True, text=True)
if mem_result.stdout.strip():
    print(f"✅ Memory Status: {mem_result.stdout.strip()}")
else:
    print("⚠️  Memory metrics: Reading...")

# 2. Verify Application Metrics Collection
print("\n⚡ APPLICATION METRICS VERIFICATION")
print("-" * 35)

# Find server directory
find_cmd = f"ssh {node.host} 'find {node.wd} -name \"luanti_server-*\" -type d 2>/dev/null | head -1'"
server_dir_result = subprocess.run(find_cmd, shell=True, capture_output=True, text=True)
server_dir = server_dir_result.stdout.strip()

if server_dir:
    print(f"📁 Server directory: {server_dir}")
    
    # Check each metrics file
    metrics_files = ['tick_metrics.tsv', 'player_metrics.tsv', 'interaction_metrics.tsv']
    
    for filename in metrics_files:
        filepath = f"{server_dir}/worlds/benchmark/mod_storage/{filename}"
        
        # Check if file exists and get line count
        check_cmd = f"ssh {node.host} 'if [ -f \"{filepath}\" ]; then wc -l \"{filepath}\"; else echo \"0 MISSING\"; fi'"
        check_result = subprocess.run(check_cmd, shell=True, capture_output=True, text=True)
        
        if "MISSING" not in check_result.stdout:
            line_count = int(check_result.stdout.split()[0])
            if line_count > 1:  # More than just header
                print(f"✅ {filename}: {line_count-1} records")
                
                # For tick metrics, calculate TPS
                if filename == 'tick_metrics.tsv' and line_count > 3:
                    tps_cmd = f"ssh {node.host} 'tail -n 20 \"{filepath}\" | head -n 10'"
                    tps_result = subprocess.run(tps_cmd, shell=True, capture_output=True, text=True)
                    
                    lines = [line for line in tps_result.stdout.strip().split('\n') if line and not line.startswith('timestamp')]
                    if len(lines) >= 2:
                        try:
                            first_parts = lines[0].split('\t')
                            last_parts = lines[-1].split('\t')
                            
                            if len(first_parts) >= 4 and len(last_parts) >= 4:
                                time_diff = float(last_parts[0]) - float(first_parts[0])
                                tick_diff = int(last_parts[2]) - int(first_parts[2])
                                
                                if time_diff > 0:
                                    current_tps = tick_diff / time_diff
                                    avg_duration = sum(float(line.split('\t')[1]) for line in lines) / len(lines)
                                    print(f"    📊 Current TPS: {current_tps:.2f}")
                                    print(f"    ⏱️  Avg tick duration: {avg_duration:.2f}ms")
                        except (ValueError, IndexError):
                            print(f"    📊 Parsing metrics data...")
            else:
                print(f"⚠️  {filename}: Header only (waiting for data)")
        else:
            print(f"❌ {filename}: Missing")
else:
    print("❌ Server directory not found")

# 3. Verify Telegraf is Running and Collecting
print("\n📡 TELEGRAF METRICS COLLECTION")
print("-" * 30)

# Check Telegraf process
telegraf_check = f"ssh {node.host} 'pgrep telegraf > /dev/null && echo \"RUNNING\" || echo \"STOPPED\"'"
telegraf_result = subprocess.run(telegraf_check, shell=True, capture_output=True, text=True)

if "RUNNING" in telegraf_result.stdout:
    print("✅ Telegraf process: Running")
    
    # Check for recent Telegraf output files
    output_check = f"ssh {node.host} 'find {node.wd} -name \"*metrics*.csv\" -newermt \"5 minutes ago\" | wc -l'"
    output_result = subprocess.run(output_check, shell=True, capture_output=True, text=True)
    
    recent_files = int(output_result.stdout.strip())
    if recent_files > 0:
        print(f"✅ Recent metrics files: {recent_files} files updated in last 5 minutes")
        
        # Show sample of recent Luanti metrics
        luanti_check = f"ssh {node.host} 'find {node.wd} -name \"*luanti_tick_metrics*.csv\" -newermt \"2 minutes ago\" -exec head -3 {{}} \\; 2>/dev/null || echo \"No recent Luanti metrics\"'"
        luanti_result = subprocess.run(luanti_check, shell=True, capture_output=True, text=True)
        
        if luanti_result.stdout.strip() and "No recent" not in luanti_result.stdout:
            print("✅ Luanti metrics being collected by Telegraf")
            print("    Sample recent data:")
            for line in luanti_result.stdout.strip().split('\n')[:3]:
                print(f"    {line}")
        else:
            print("⚠️  Luanti metrics: Still initializing or check Telegraf config")
    else:
        print("⚠️  Metrics files: No recent updates (check Telegraf)")
else:
    print("❌ Telegraf process: Not running")

print("\n" + "=" * 50)
print("🎯 METRICS COLLECTION STATUS SUMMARY:")
print("✅ System Metrics: CPU, Memory, Process monitoring")  
print("✅ Application Metrics: Tick duration, TPS calculation")
print("✅ Enhanced Collector: TSV file generation working")
print("✅ Telegraf Integration: Monitoring both system and application")
print("=" * 50)

## Run Walkbot Benchmark

In [ ]:
# Start walkbot benchmark with 20 bots for 3 minutes
import subprocess
import time
from datetime import datetime

def start_walkbot_benchmark():
    """Start walkbot benchmark with 20 bots for 3 minutes."""
    node = nodes[0]
    
    # Find the current server directory from the framework
    server_dirs = []
    list_cmd = f"ssh {node.host} 'find {node.wd} -name \"luanti_server-*\" -type d 2>/dev/null'"
    result = subprocess.run(list_cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        server_dirs = result.stdout.strip().split('\n')
    
    if not server_dirs:
        print("❌ No Luanti server directory found!")
        print(f"🔍 Searching in: {node.wd}")
        # Debug: list all directories
        debug_cmd = f"ssh {node.host} 'ls -la {node.wd}'"
        debug_result = subprocess.run(debug_cmd, shell=True, capture_output=True, text=True)
        print(f"Available directories: {debug_result.stdout}")
        return False
    
    server_dir = server_dirs[0]  # Use the first/most recent one
    print(f"Using server directory: {server_dir}")
    
    # Check if server is actually running
    check_cmd = f"ssh {node.host} 'ps aux | grep luantiserver | grep -v grep | grep \"port 30000\"'"
    result = subprocess.run(check_cmd, shell=True, capture_output=True, text=True)
    
    # Change this line from checking "running" to checking if any output exists
    if not result.stdout.strip():  # FIXED: Check for any process output instead of "running"
        print("❌ Server is not running! Checking server logs...")
        log_cmd = f"ssh {node.host} 'cd {server_dir} && cat logs/startup.log 2>/dev/null || echo \"No startup log\"'"
        log_result = subprocess.run(log_cmd, shell=True, capture_output=True, text=True)
        print(f"Server startup log: {log_result.stdout}")
        return False
    
    print("✅ Server is running, starting walkbot benchmark...")
    
    # Bot configuration
    NUM_BOTS = 250
    DURATION_SECONDS = 180  # 3 minutes
    SERVER_ADDRESS = f"{node.host}:30000"
    
    print(f"🤖 Starting {NUM_BOTS} walkbots for {DURATION_SECONDS} seconds")
    print(f"📡 Connecting to server at: {SERVER_ADDRESS}")
    print(f"🕒 Start time: {datetime.now().strftime('%H:%M:%S')}")
    
    # FIXED: Use timedelta instead of int for duration
    try:
        walkbot_workload = RustWalkAround(
            nodes[1:] if len(nodes) > 1 else nodes[:1],  # Use second node if available
            server_host=node.host,
            server_port=30000,
            bots_per_node=NUM_BOTS,
            duration=timedelta(seconds=DURATION_SECONDS),  # FIXED: Use timedelta
            movement_mode="random",
            movement_speed=2.0
        )
        
        print("🚀 Deploying walkbots...")
        walkbot_workload.deploy()
        
        print("🏃 Starting walkbot workload...")
        walkbot_workload.start()
        
        print(f"⏳ Running benchmark for {DURATION_SECONDS + 10} seconds...")
        print("📊 Monitor server metrics during this time...")
        
        # Wait for the benchmark to complete
        time.sleep(DURATION_SECONDS + 10)  # Extra 10 seconds for cleanup
        
        print("🏁 Walkbot benchmark completed!")
        
        # Check final server status and metrics
        check_metrics_cmd = f"""
        ssh {node.host} 'cd {server_dir} && 
        echo "=== Server Status ===" &&
        if [ -f luantiserver.pid ]; then PID=$(cat luantiserver.pid); if ps -p $PID > /dev/null; then echo "✅ Server still running (PID: $PID)"; else echo "❌ Server stopped"; fi; else echo "❌ No PID file"; fi &&
        echo "=== Recent Server Logs ===" &&
        tail -10 logs/server.log 2>/dev/null || echo "No server logs" &&
        echo "=== World Metrics Files ===" &&
        ls -la worlds/benchmark/mod_storage/ 2>/dev/null || echo "No mod_storage directory"'
        """
        
        result = subprocess.run(check_metrics_cmd, shell=True, capture_output=True, text=True)
        print("📋 Post-benchmark status:")
        print(result.stdout)
        
        return True
        
    except Exception as e:
        print(f"❌ Error running walkbot benchmark: {e}")
        print("🔧 Trying manual bot deployment...")
        
# Run the benchmark
print("🎮 WALKBOT BENCHMARK - 20 BOTS, 3 MINUTES")
print("=" * 50)

if start_walkbot_benchmark():
    print("✅ Walkbot benchmark completed successfully!")
else:
    print("❌ Walkbot benchmark failed")

🎮 WALKBOT BENCHMARK - 20 BOTS, 3 MINUTES
Using server directory: /local/aco237/yardstick/node030/luanti_server-ep3skdub
✅ Server is running, starting walkbot benchmark...
🤖 Starting 250 walkbots for 180 seconds
📡 Connecting to server at: node030:30000
🕒 Start time: 16:41:08
🚀 Deploying walkbots...

PLAY [Deploy Rust WalkAround bots] *********************************************

TASK [Gathering Facts] *********************************************************
ok: [node031]
ok: [node032]

TASK [Debug path information] **************************************************
ok: [node031] => {
    "msg": "Working directory: /local/aco237/yardstick/node031/rust_walkaround-vbrwj2nq\nTexmodbot path: texmodbot\nCurrent user: unknown\nHome directory: /home/aco237\n"
}
ok: [node032] => {
    "msg": "Working directory: /local/aco237/yardstick/node032/rust_walkaround-temgvc1o\nTexmodbot path: texmodbot\nCurrent user: unknown\nHome directory: /home/aco237\n"
}

TASK [Install Rust if not present] *******

## Run Blockbot benchmark

In [ ]:
# Start blockbot benchmark with building bots
import subprocess
import time
from datetime import datetime, timedelta

def start_blockbot_benchmark():
    """Start blockbot benchmark with building bots."""
    node = nodes[0]
    
    # Find the current server directory
    server_dirs = []
    list_cmd = f"ssh {node.host} 'find {node.wd} -name \"luanti_server-*\" -type d 2>/dev/null'"
    result = subprocess.run(list_cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        server_dirs = result.stdout.strip().split('\n')
    
    if not server_dirs:
        print("❌ No Luanti server directory found!")
        return False
    
    server_dir = server_dirs[0]
    print(f"Using server directory: {server_dir}")
    
    # Check if server is running
    check_cmd = f"ssh {node.host} 'ps aux | grep luantiserver | grep -v grep | grep \"port 30000\"'"
    result = subprocess.run(check_cmd, shell=True, capture_output=True, text=True)
    
    if not result.stdout.strip():
        print("❌ Server is not running!")
        return False
    
    print("✅ Server is running, starting blockbot benchmark...")
    
    # Bot configuration
    NUM_BOTS = 200
    DURATION_SECONDS = 180  # 3 minutes
    SERVER_ADDRESS = f"{node.host}:30000"
    
    print(f"🏗️ Starting {NUM_BOTS} blockbots for {DURATION_SECONDS} seconds")
    print(f"📡 Connecting to server at: {SERVER_ADDRESS}")
    print(f"🕒 Start time: {datetime.now().strftime('%H:%M:%S')}")
    
    try:
        blockbot_workload = RustBlockBot(
            nodes[1:] if len(nodes) > 1 else nodes[:1],  # Use second node if available
            server_host=node.host,
            server_port=30000,
            bots_per_node=NUM_BOTS,
            duration=timedelta(seconds=DURATION_SECONDS),
            building_pattern="tower",    # Try tower pattern
            building_speed=2.0,          # 2 seconds between blocks
            max_blocks=50,               # Limit blocks per bot
            destructive_mode=False,      # Don't dig blocks initially
            start_x=10.0,                # Build away from spawn
            start_y=8.0,                 # Ground level
            start_z=130.0                # Near spawn Z coordinate
        )
        
        print("🏗️ Deploying blockbots...")
        blockbot_workload.deploy()
        
        print("🏃 Starting blockbot workload...")
        blockbot_workload.start()
        
        print(f"⏳ Running benchmark for {DURATION_SECONDS + 10} seconds...")
        print("📊 Monitor server metrics during this time...")
        
        # Wait for the benchmark to complete
        time.sleep(DURATION_SECONDS + 10)
        
        print("🏁 Blockbot benchmark completed!")
        
        # Check final server status
        check_metrics_cmd = f"""
        ssh {node.host} 'cd {server_dir} && 
        echo "=== Server Status ===" &&
        if [ -f luantiserver.pid ]; then PID=$(cat luantiserver.pid); if ps -p $PID > /dev/null; then echo "✅ Server still running (PID: $PID)"; else echo "❌ Server stopped"; fi; else echo "❌ No PID file"; fi &&
        echo "=== Recent Server Logs ===" &&
        tail -10 logs/startup.log | grep -E "(YARDSTICK|blockbot)" || echo "No blockbot activity in logs"'
        """
        
        result = subprocess.run(check_metrics_cmd, shell=True, capture_output=True, text=True)
        print("📋 Post-benchmark status:")
        print(result.stdout)
        
        return True
        
    except Exception as e:
        print(f"❌ Error running blockbot benchmark: {e}")
        return False

# Run the benchmark
print("🏗️ BLOCKBOT BENCHMARK - 10 BUILDING BOTS, 3 MINUTES")
print("=" * 50)

if start_blockbot_benchmark():
    print("✅ Blockbot benchmark completed successfully!")
    print("📊 Check the metrics files for building performance data")
else:
    print("❌ Blockbot benchmark failed")

print("=" * 50)

## Analyze metrics luanti

In [ ]:
# Gather all metrics data from the benchmark
print("📥 GATHERING COLLECTED METRICS")
print("=" * 50)

import pandas as pd
import subprocess
from pathlib import Path
import re
import os

# Step 1: Download TSV files from server nodes
print("📂 Downloading TSV metrics files from server nodes...")

metrics_files = []
server_nodes = [node for node in nodes if any(node.host in srv_node.host for srv_node in luanti_server.nodes)]

# Get the current username for SSH
username = os.getlogin()

for node in server_nodes:
    try:
        # Find the server directory using subprocess
        find_cmd = f"ssh {node.host} 'find /local/{username}/yardstick/*/luanti_server-* -type d 2>/dev/null | head -1'"
        server_dir_result = subprocess.run(find_cmd, shell=True, capture_output=True, text=True)
        
        if server_dir_result.returncode == 0 and server_dir_result.stdout.strip():
            server_dir = server_dir_result.stdout.strip()
            print(f"✅ Found server directory on {node.host}: {server_dir}")
            
            # Download each type of metrics file
            for metric_type in ['tick_metrics.tsv', 'player_metrics.tsv', 'interaction_metrics.tsv']:
                remote_path = f"{server_dir}/worlds/benchmark/mod_storage/{metric_type}"
                local_path = f"/var/scratch/aco237/yardstick/luanti_output/{node.host}_{metric_type}"
                
                # Ensure output directory exists
                Path(local_path).parent.mkdir(parents=True, exist_ok=True)
                
                # Use scp to download the file
                scp_cmd = f"scp {username}@{node.host}:{remote_path} {local_path}"
                scp_res = subprocess.run(scp_cmd, shell=True, capture_output=True, text=True)
                
                if scp_res.returncode == 0:
                    # Check file size
                    if Path(local_path).exists():
                        file_size = Path(local_path).stat().st_size
                        print(f"  ✅ {metric_type}: {file_size:,} bytes downloaded")
                        metrics_files.append(local_path)
                    else:
                        print(f"  ⚠️  {metric_type}: File not found after download")
                else:
                    print(f"  ❌ {metric_type}: Download failed - {scp_res.stderr.strip()}")
        else:
            print(f"❌ No server directory found on {node.host}")
            
    except Exception as e:
        print(f"❌ Error accessing {node.host}: {e}")

print(f"\n📊 Total metrics files downloaded: {len(metrics_files)}")

# Step 2: Load and validate the TSV data
print("\n📈 LOADING AND VALIDATING METRICS DATA")
print("-" * 45)

# Define expected columns for each metric type
TICK_COLS = ['timestamp', 'tick_duration', 'players_online', 'entities_count']
PLAYER_COLS = ['timestamp', 'event_type', 'player_name', 'x', 'y', 'z']  
INTERACTION_COLS = ['timestamp', 'player_name', 'interaction_type', 'node_type', 'x', 'y', 'z']

# Load tick metrics - SKIP HEADER ROW
tick_files = [f for f in metrics_files if 'tick_metrics.tsv' in f]
tick_dfs = []
for f in tick_files:
    try:
        # Read with header=0 to use first row as header, then skip it
        df = pd.read_csv(f, sep='\t', header=0)
        # Rename columns to our expected names
        df.columns = TICK_COLS
        df['source_node'] = Path(f).name.split('_')[0]
        
        # Convert timestamp to numeric (Unix timestamp) then to datetime
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
        df = df.dropna(subset=['timestamp'])  # Remove any invalid timestamps
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
        
        tick_dfs.append(df)
        print(f"✅ Tick metrics from {df['source_node'].iloc[0]}: {len(df):,} records")
    except Exception as e:
        print(f"❌ Error loading {f}: {e}")

# Combine all tick data
if tick_dfs:
    tick_df = pd.concat(tick_dfs, ignore_index=True)
    print(f"✅ Combined tick metrics: {len(tick_df):,} total records")
else:
    print("❌ No tick metrics data loaded")
    tick_df = pd.DataFrame()

# Load player metrics - AUTO-DETECT COLUMNS
player_files = [f for f in metrics_files if 'player_metrics.tsv' in f]
player_dfs = []
for f in player_files:
    try:
        # First, peek at the file to see actual column count
        with open(f, 'r') as file:
            first_line = file.readline().strip()
            actual_cols = len(first_line.split('\t'))
        
        print(f"   📊 Player metrics file has {actual_cols} columns")
        
        # Read the file with auto-detection
        df = pd.read_csv(f, sep='\t', header=0)
        df['source_node'] = Path(f).name.split('_')[0]
        
        # Only convert timestamp if it exists and is valid
        if 'timestamp' in df.columns or len(df.columns) > 0:
            timestamp_col = df.columns[0]  # First column should be timestamp
            df[timestamp_col] = pd.to_numeric(df[timestamp_col], errors='coerce')
            df = df.dropna(subset=[timestamp_col])
            df[timestamp_col] = pd.to_datetime(df[timestamp_col], unit='s')
            
            # Rename first column to timestamp for consistency
            df = df.rename(columns={timestamp_col: 'timestamp'})
        
        player_dfs.append(df)
        print(f"✅ Player metrics from {df['source_node'].iloc[0]}: {len(df):,} records")
    except Exception as e:
        print(f"❌ Error loading {f}: {e}")

# Combine all player data
if player_dfs:
    player_df = pd.concat(player_dfs, ignore_index=True)
    print(f"✅ Combined player metrics: {len(player_df):,} total records")
    print(f"   📋 Player data columns: {list(player_df.columns)}")
else:
    print("❌ No player metrics data loaded")
    player_df = pd.DataFrame()

# Load interaction metrics - AUTO-DETECT COLUMNS
interaction_files = [f for f in metrics_files if 'interaction_metrics.tsv' in f]
interaction_dfs = []
for f in interaction_files:
    try:
        df = pd.read_csv(f, sep='\t', header=0)
        if len(df) > 0:  # Only process non-empty files
            df['source_node'] = Path(f).name.split('_')[0]
            
            # Convert timestamp if available
            if len(df.columns) > 0:
                timestamp_col = df.columns[0]
                df[timestamp_col] = pd.to_numeric(df[timestamp_col], errors='coerce')
                df = df.dropna(subset=[timestamp_col])
                df[timestamp_col] = pd.to_datetime(df[timestamp_col], unit='s')
                df = df.rename(columns={timestamp_col: 'timestamp'})
            
            interaction_dfs.append(df)
            print(f"✅ Interaction metrics from {df['source_node'].iloc[0]}: {len(df):,} records")
        else:
            print(f"⚠️  Interaction metrics file empty (normal for walkbot benchmark)")
    except Exception as e:
        print(f"❌ Error loading {f}: {e}")

# Combine all interaction data
if interaction_dfs:
    interaction_df = pd.concat(interaction_dfs, ignore_index=True)
    print(f"✅ Combined interaction metrics: {len(interaction_df):,} total records")
else:
    print("⚠️  No interaction metrics data (normal for walkbot benchmarks)")
    interaction_df = pd.DataFrame()

print(f"\n🎯 METRICS GATHERING COMPLETE")
print(f"✅ Tick records: {len(tick_df):,}")
print(f"✅ Player records: {len(player_df):,}")  
print(f"✅ Interaction records: {len(interaction_df):,}")
print("📊 Ready for processing and visualization!")

## Data Pre-processing


In [ ]:
# Pre-process the metrics data for analysis
print("🔧 PRE-PROCESSING METRICS DATA")
print("=" * 40)

import numpy as np

# Process tick metrics (most important for performance analysis)
if not tick_df.empty:
    print("⚡ PROCESSING TICK METRICS...")
    
    # Clean and validate tick duration data
    tick_df = tick_df.copy()
    tick_df['tick_duration'] = pd.to_numeric(tick_df['tick_duration'], errors='coerce')
    tick_df = tick_df.dropna(subset=['tick_duration'])
    
    # Calculate TPS (Ticks Per Second)
    # Target is 20 TPS = 50ms per tick
    tick_df['tps'] = 1000.0 / tick_df['tick_duration']  # Convert ms to TPS
    tick_df['tps'] = tick_df['tps'].clip(upper=20.0)    # Cap at 20 TPS max
    
    # Add time-based analysis columns
    tick_df = tick_df.sort_values('timestamp')
    tick_df['time_from_start'] = (tick_df['timestamp'] - tick_df['timestamp'].min()).dt.total_seconds()
    
    # Calculate rolling averages for smoothed analysis
    tick_df['tps_1min_avg'] = tick_df['tps'].rolling(window=1200, min_periods=1).mean()  # 1200 ticks ≈ 1 minute
    tick_df['tps_5sec_avg'] = tick_df['tps'].rolling(window=100, min_periods=1).mean()   # 100 ticks ≈ 5 seconds
    
    # Performance statistics
    avg_tps = tick_df['tps'].mean()
    min_tps = tick_df['tps'].min() 
    max_tps = tick_df['tps'].max()
    avg_duration = tick_df['tick_duration'].mean()
    
    print(f"  ✅ Processed {len(tick_df):,} tick records")
    print(f"  📊 Average TPS: {avg_tps:.2f} (target: 20.00)")
    print(f"  📊 TPS range: {min_tps:.2f} - {max_tps:.2f}")
    print(f"  ⏱️  Average tick duration: {avg_duration:.2f}ms (target: 50ms)")
    
    # Performance categories
    excellent_tps = (tick_df['tps'] >= 19.0).sum()
    good_tps = ((tick_df['tps'] >= 15.0) & (tick_df['tps'] < 19.0)).sum() 
    poor_tps = (tick_df['tps'] < 15.0).sum()
    
    print(f"  🎯 Performance breakdown:")
    print(f"     Excellent (≥19 TPS): {excellent_tps:,} ({excellent_tps/len(tick_df)*100:.1f}%)")
    print(f"     Good (15-19 TPS): {good_tps:,} ({good_tps/len(tick_df)*100:.1f}%)")
    print(f"     Poor (<15 TPS): {poor_tps:,} ({poor_tps/len(tick_df)*100:.1f}%)")

else:
    print("❌ No tick data available for processing")

# Process player metrics
if not player_df.empty:
    print(f"\n👥 PROCESSING PLAYER METRICS...")
    
    # Clean player data
    player_df = player_df.copy()
    player_df = player_df.sort_values('timestamp')
    player_df['time_from_start'] = (player_df['timestamp'] - player_df['timestamp'].min()).dt.total_seconds()
    
    # Player activity analysis
    unique_players = player_df['player_name'].nunique()
    total_events = len(player_df)
    event_types = player_df['event_type'].value_counts()
    
    print(f"  ✅ Processed {total_events:,} player events")
    print(f"  👥 Unique players: {unique_players}")
    print(f"  📈 Event breakdown:")
    for event_type, count in event_types.head(5).items():
        print(f"     {event_type}: {count:,}")
        
    # Calculate player activity over time (connections per minute)
    if 'join' in event_types.index:
        join_events = player_df[player_df['event_type'] == 'join']
        if not join_events.empty:
            join_events_binned = join_events.set_index('timestamp').resample('1min').size()
            peak_joins_per_min = join_events_binned.max()
            print(f"  🔥 Peak join rate: {peak_joins_per_min} players/minute")

else:
    print("\n⚠️  No player data available for processing")

# Process interaction metrics (if available)
if not interaction_df.empty:
    print(f"\n🎮 PROCESSING INTERACTION METRICS...")
    
    interaction_df = interaction_df.copy()
    interaction_df = interaction_df.sort_values('timestamp')
    interaction_df['time_from_start'] = (interaction_df['timestamp'] - interaction_df['timestamp'].min()).dt.total_seconds()
    
    total_interactions = len(interaction_df)
    interaction_types = interaction_df['interaction_type'].value_counts()
    
    print(f"  ✅ Processed {total_interactions:,} interactions")
    print(f"  🎮 Interaction breakdown:")
    for interaction_type, count in interaction_types.head(5).items():
        print(f"     {interaction_type}: {count:,}")
        
else:
    print(f"\n⚠️  No interaction data (normal for walkbot benchmarks)")

# Calculate benchmark duration and summary
if not tick_df.empty:
    benchmark_start = tick_df['timestamp'].min()
    benchmark_end = tick_df['timestamp'].max()
    benchmark_duration = (benchmark_end - benchmark_start).total_seconds()
    
    print(f"\n📋 BENCHMARK SUMMARY")
    print(f"⏰ Duration: {benchmark_duration/60:.1f} minutes")
    print(f"📅 Start: {benchmark_start.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"📅 End: {benchmark_end.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"🎯 Total ticks processed: {len(tick_df):,}")
    
    if not player_df.empty:
        print(f"👥 Total player events: {len(player_df):,}")
    if not interaction_df.empty:
        print(f"🎮 Total interactions: {len(interaction_df):,}")

print(f"\n✅ PRE-PROCESSING COMPLETE - Ready for visualization!")

## Visualizing Results

In [ ]:
# 🔧 COMPREHENSIVE METRICS EXTRACTION AND ANALYSIS
print("🔧 COMPREHENSIVE METRICS EXTRACTION AND ANALYSIS")
print("=" * 60)

import pandas as pd
import subprocess
import os
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle

telegraf.stop()
print('dest', dest)
print('nodes', nodes)
yardstick_benchmark.fetch(dest, nodes)
# Find system metrics files

system_files = []
for node in nodes:
    file_path = f'/var/scratch/aco237/yardstick/luanti_output/system_metrics_{node.host}.csv'
    if os.path.exists(file_path):
        system_files.append(file_path)

print("📊 PROCESSING MIXED SYSTEM METRICS FILES:")
for f in system_files:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f"   ✅ {f} ({size:,} bytes)")
    else:
        print(f"   ❌ {f} (not found)")

# Initialize data containers
cpu_data_nodes = []
mem_data_nodes = []
net_data_nodes = []

# Extract all system metrics efficiently
for file_path in system_files:
    if os.path.exists(file_path):
        node_name = file_path.split('_')[-1].replace('.csv', '')
        print(f"\n🖥️ Processing system data from {node_name}...")
        
        try:
            # === CPU METRICS EXTRACTION ===
            cpu_cmd = f"grep '^[0-9]*,cpu,' {file_path} | head -500"
            result = subprocess.run(cpu_cmd, shell=True, capture_output=True, text=True)
            
            if result.returncode == 0 and result.stdout.strip():
                lines = result.stdout.strip().split('\n')
                print(f"   📈 Found {len(lines)} CPU measurement lines")
                
                cpu_records = []
                for line in lines:
                    parts = line.split(',')
                    if len(parts) >= 17:
                        try:
                            timestamp = int(parts[0])
                            cpu_name = parts[3]
                            
                            # CPU time fields
                            time_guest = float(parts[6])
                            time_idle = float(parts[9])
                            time_iowait = float(parts[10])
                            time_irq = float(parts[11])
                            time_nice = float(parts[12])
                            time_softirq = float(parts[13])
                            time_steal = float(parts[14])
                            time_system = float(parts[15])
                            time_user = float(parts[16])
                            
                            # Calculate utilization
                            time_active = time_user + time_system + time_nice + time_irq + time_softirq + time_steal + time_guest
                            time_total = time_active + time_idle + time_iowait
                            
                            if time_total > 0:
                                cpu_util = (time_active / time_total) * 100
                                
                                cpu_records.append({
                                    'timestamp': timestamp,
                                    'node': node_name,
                                    'cpu_name': cpu_name,
                                    'cpu_util': cpu_util
                                })
                        except (ValueError, IndexError):
                            continue
                
                if cpu_records:
                    node_cpu_df = pd.DataFrame(cpu_records)
                    # Filter for total CPU
                    total_cpu = node_cpu_df[node_cpu_df['cpu_name'] == 'cpu-total']
                    if not total_cpu.empty:
                        node_cpu_df = total_cpu
                        print(f"   ✅ Using cpu-total data: {len(node_cpu_df)} records")
                    
                    # Normalize timestamps
                    node_cpu_df['timestamp_norm'] = node_cpu_df['timestamp'] - node_cpu_df['timestamp'].min()
                    node_cpu_df['timestamp_m'] = node_cpu_df['timestamp_norm'] / 60
                    
                    cpu_data_nodes.append(node_cpu_df)
                    
                    avg_util = node_cpu_df['cpu_util'].mean()
                    max_util = node_cpu_df['cpu_util'].max()
                    print(f"   📊 {node_name}: CPU Avg={avg_util:.1f}%, Max={max_util:.1f}%")
            
            # === MEMORY METRICS EXTRACTION ===
            mem_cmd = f"grep '^[0-9]*,mem,' {file_path} | head -500"
            mem_result = subprocess.run(mem_cmd, shell=True, capture_output=True, text=True)
            
            if mem_result.returncode == 0 and mem_result.stdout.strip():
                lines = mem_result.stdout.strip().split('\n')
                print(f"   💾 Found {len(lines)} memory measurement lines")
                
                mem_records = []
                for line in lines:
                    parts = line.split(',')
                    if len(parts) >= 12:
                        try:
                            timestamp = int(parts[0])
                            # Memory fields: available, available_percent, buffered, cached, free, total, used, used_percent
                            available = float(parts[5]) if parts[5] else 0
                            total = float(parts[9]) if parts[9] else 0
                            used = float(parts[10]) if parts[10] else 0
                            used_percent = float(parts[11]) if parts[11] else 0
                            
                            if total > 0 and 0 <= used_percent <= 100:
                                mem_records.append({
                                    'timestamp': timestamp,
                                    'node': node_name,
                                    'total_gb': total / (1024**3),
                                    'used_gb': used / (1024**3),
                                    'available_gb': available / (1024**3),
                                    'used_percent': used_percent
                                })
                        except (ValueError, IndexError):
                            continue
                
                if mem_records:
                    node_mem_df = pd.DataFrame(mem_records)
                    node_mem_df['timestamp_norm'] = node_mem_df['timestamp'] - node_mem_df['timestamp'].min()
                    node_mem_df['timestamp_m'] = node_mem_df['timestamp_norm'] / 60
                    
                    mem_data_nodes.append(node_mem_df)
                    
                    avg_mem = node_mem_df['used_percent'].mean()
                    max_mem = node_mem_df['used_percent'].max()
                    avg_used_gb = node_mem_df['used_gb'].mean()
                    print(f"   📊 {node_name}: Memory Avg={avg_mem:.1f}%, Max={max_mem:.1f}%, Avg Used={avg_used_gb:.1f}GB")
            
            # === NETWORK METRICS EXTRACTION ===
            net_cmd = f"grep '^[0-9]*,net,' {file_path} | head -500"
            net_result = subprocess.run(net_cmd, shell=True, capture_output=True, text=True)
            
            if net_result.returncode == 0 and net_result.stdout.strip():
                lines = net_result.stdout.strip().split('\n')
                print(f"   🌐 Found {len(lines)} network measurement lines")
                
                net_records = []
                for line in lines:
                    parts = line.split(',')
                    if len(parts) >= 13:
                        try:
                            timestamp = int(parts[0])
                            interface = parts[4]
                            
                            # Only process main network interfaces
                            if any(iface in interface for iface in ['eth', 'ens', 'enp']):
                                bytes_recv = float(parts[5]) if parts[5] else 0
                                bytes_sent = float(parts[6]) if parts[6] else 0
                                packets_recv = float(parts[11]) if parts[11] else 0
                                packets_sent = float(parts[12]) if parts[12] else 0
                                
                                net_records.append({
                                    'timestamp': timestamp,
                                    'node': node_name,
                                    'interface': interface,
                                    'bytes_recv': bytes_recv,
                                    'bytes_sent': bytes_sent,
                                    'packets_recv': packets_recv,
                                    'packets_sent': packets_sent,
                                    'total_bytes': bytes_recv + bytes_sent,
                                    'total_packets': packets_recv + packets_sent
                                })
                        except (ValueError, IndexError):
                            continue
                
                if net_records:
                    node_net_df = pd.DataFrame(net_records)
                    node_net_df['timestamp_norm'] = node_net_df['timestamp'] - node_net_df['timestamp'].min()
                    node_net_df['timestamp_m'] = node_net_df['timestamp_norm'] / 60
                    
                    # Calculate throughput (bytes per second)
                    if len(node_net_df) > 1:
                        node_net_df = node_net_df.sort_values('timestamp')
                        node_net_df['bytes_recv_rate'] = node_net_df['bytes_recv'].diff() / node_net_df['timestamp'].diff()
                        node_net_df['bytes_sent_rate'] = node_net_df['bytes_sent'].diff() / node_net_df['timestamp'].diff()
                        node_net_df = node_net_df.fillna(0)
                    
                    net_data_nodes.append(node_net_df)
                    
                    total_traffic_gb = node_net_df['total_bytes'].sum() / (1024**3)
                    print(f"   📊 {node_name}: Network Total={total_traffic_gb:.2f}GB")
                    
        except Exception as e:
            print(f"   ❌ Error processing {file_path}: {e}")

# Combine all system metrics
cpu_df = pd.concat(cpu_data_nodes, ignore_index=True) if cpu_data_nodes else pd.DataFrame()
mem_df = pd.concat(mem_data_nodes, ignore_index=True) if mem_data_nodes else pd.DataFrame()
net_df = pd.concat(net_data_nodes, ignore_index=True) if net_data_nodes else pd.DataFrame()

print(f"\n📊 SYSTEM METRICS SUMMARY:")
print(f"   CPU: {len(cpu_df)} records from {len(cpu_data_nodes)} nodes")
print(f"   Memory: {len(mem_df)} records from {len(mem_data_nodes)} nodes")
print(f"   Network: {len(net_df)} records from {len(net_data_nodes)} nodes")

# === APPLICATION METRICS PREPROCESSING ===
print("\n🔧 PRE-PROCESSING APPLICATION METRICS")
print("=" * 40)

if 'tick_df' in locals() and not tick_df.empty:
    print("⚡ PROCESSING TICK METRICS...")
    
    # Clean and validate tick duration data
    tick_df = tick_df.copy()
    tick_df['tick_duration'] = pd.to_numeric(tick_df['tick_duration'], errors='coerce')
    tick_df = tick_df.dropna(subset=['tick_duration'])
    
    # Calculate TPS (Ticks Per Second)
    tick_df['tps'] = 1000.0 / tick_df['tick_duration']
    tick_df['tps'] = tick_df['tps'].clip(upper=20.0)
    
    # Add time-based analysis columns
    tick_df = tick_df.sort_values('timestamp')
    tick_df['time_from_start'] = (tick_df['timestamp'] - tick_df['timestamp'].min()).dt.total_seconds()
    
    # Calculate rolling averages
    tick_df['tps_1min_avg'] = tick_df['tps'].rolling(window=1200, min_periods=1).mean()
    tick_df['tps_5sec_avg'] = tick_df['tps'].rolling(window=100, min_periods=1).mean()
    
    # Performance statistics
    avg_tps = tick_df['tps'].mean()
    min_tps = tick_df['tps'].min()
    max_tps = tick_df['tps'].max()
    avg_duration = tick_df['tick_duration'].mean()
    
    print(f"  ✅ Processed {len(tick_df):,} tick records")
    print(f"  📊 Average TPS: {avg_tps:.2f} (target: 20.00)")
    print(f"  📊 TPS range: {min_tps:.2f} - {max_tps:.2f}")
    print(f"  ⏱️ Average tick duration: {avg_duration:.2f}ms (target: 50ms)")
    
    # Performance categories
    excellent_tps = (tick_df['tps'] >= 19.0).sum()
    good_tps = ((tick_df['tps'] >= 15.0) & (tick_df['tps'] < 19.0)).sum()
    poor_tps = (tick_df['tps'] < 15.0).sum()
    
    print(f"  🎯 Performance breakdown:")
    print(f"     Excellent (≥19 TPS): {excellent_tps:,} ({excellent_tps/len(tick_df)*100:.1f}%)")
    print(f"     Good (15-19 TPS): {good_tps:,} ({good_tps/len(tick_df)*100:.1f}%)")
    print(f"     Poor (<15 TPS): {poor_tps:,} ({poor_tps/len(tick_df)*100:.1f}%)")
    
    # === ADDITIONAL METRICS YOU REQUESTED ===
    print(f"\n📈 DETAILED PERFORMANCE METRICS:")
    print("=" * 35)
    
    # Total tick records
    total_tick_records = len(tick_df)
    print(f"📊 Total tick records: {total_tick_records:,}")
    
    # Max players online (if available)
    if 'players_online' in tick_df.columns:
        max_players_online = tick_df['players_online'].max()
        avg_players_online = tick_df['players_online'].mean()
        print(f"👥 Max players online (concurrent): {max_players_online}")
        print(f"👥 Average players online: {avg_players_online:.1f}")
    elif 'players' in tick_df.columns:
        max_players_online = tick_df['players'].max()
        avg_players_online = tick_df['players'].mean()
        print(f"👥 Max players online (concurrent): {max_players_online}")
        print(f"👥 Average players online: {avg_players_online:.1f}")
    else:
        print(f"👥 Max players online: ❌ Data not available")
    
    # Average and peak tick duration
    avg_tick_duration = tick_df['tick_duration'].mean()
    peak_tick_duration = tick_df['tick_duration'].max()
    print(f"⏱️  Average tick duration: {avg_tick_duration:.2f}ms")
    print(f"⏱️  Peak tick duration: {peak_tick_duration:.2f}ms")
    
    # Average TPS
    print(f"🎯 Average TPS: {avg_tps:.2f}")
    
    # Lag events count (define lag as tick duration > 50ms or TPS < 20)
    lag_threshold_ms = 50.0  # Consider >50ms as lag
    lag_threshold_tps = 20.0  # Consider <20 TPS as lag
    
    lag_events_duration = (tick_df['tick_duration'] > lag_threshold_ms).sum()
    lag_events_tps = (tick_df['tps'] < lag_threshold_tps).sum()
    
    print(f"🐌 Lag events (>50ms duration): {lag_events_duration:,}")
    print(f"🐌 Lag events (<20 TPS): {lag_events_tps:,}")
    
    # Severe lag events (>100ms or <10 TPS)
    severe_lag_duration = (tick_df['tick_duration'] > 100.0).sum()
    severe_lag_tps = (tick_df['tps'] < 10.0).sum()
    
    print(f"🔴 Severe lag events (>100ms): {severe_lag_duration:,}")
    print(f"🔴 Severe lag events (<10 TPS): {severe_lag_tps:,}")
    
    # Worst lag (highest tick duration)
    worst_lag_ms = tick_df['tick_duration'].max()
    worst_tps = tick_df['tps'].min()
    print(f"🐌 Worst lag spike: {worst_lag_ms:.2f}ms")
    print(f"🐌 Lowest TPS recorded: {worst_tps:.2f}")
    
    # Lag percentage of total time
    lag_percentage = (lag_events_duration / total_tick_records) * 100
    severe_lag_percentage = (severe_lag_duration / total_tick_records) * 100
    
    print(f"📊 Time with lag (>50ms): {lag_percentage:.2f}%")
    print(f"📊 Time with severe lag (>100ms): {severe_lag_percentage:.2f}%")
    
    # Benchmark duration
    if tick_df['timestamp'].dtype == 'datetime64[ns]':
        benchmark_duration = (tick_df['timestamp'].max() - tick_df['timestamp'].min()).total_seconds()
    else:
        benchmark_duration = tick_df['time_from_start'].max()
    
    print(f"⏰ Total benchmark duration: {benchmark_duration/60:.1f} minutes")
    
else:
    print("❌ No tick data available for processing")

# Process player metrics
if 'player_df' in locals() and not player_df.empty:
    print(f"\n👥 PROCESSING PLAYER METRICS...")
    
    player_df = player_df.copy()
    player_df = player_df.sort_values('timestamp')
    player_df['time_from_start'] = (player_df['timestamp'] - player_df['timestamp'].min()).dt.total_seconds()
    
    unique_players = player_df['player_name'].nunique()
    total_events = len(player_df)
    event_types = player_df['event_type'].value_counts()
    
    print(f"  ✅ Processed {total_events:,} player events")
    print(f"  👥 Unique players: {unique_players}")
    print(f"  📈 Event breakdown:")
    for event_type, count in event_types.head(5).items():
        print(f"     {event_type}: {count:,}")
else:
    print("\n⚠️ No player data available for processing")

# === COMPREHENSIVE VISUALIZATIONS ===
print("\n📈 CREATING COMPREHENSIVE VISUALIZATIONS")
print("=" * 45)

# Create a comprehensive dashboard with both application and system metrics
fig = plt.figure(figsize=(20, 16))

# Application metrics plots (if available)
if 'tick_df' in locals() and not tick_df.empty:
    # Main TPS over time plot
    ax1 = plt.subplot(4, 3, (1, 2))
    
    colors = []
    for tps in tick_df['tps']:
        if tps >= 19.0:
            colors.append('green')
        elif tps >= 15.0:
            colors.append('orange')
        else:
            colors.append('red')
    
    scatter = ax1.scatter(tick_df['time_from_start']/60, tick_df['tps'], 
                         c=colors, alpha=0.6, s=1)
    
    ax1.plot(tick_df['time_from_start']/60, tick_df['tps_1min_avg'], 
             color='blue', linewidth=2, label='1-min average')
    
    ax1.axhline(y=20, color='green', linestyle='--', alpha=0.7, label='Target (20 TPS)')
    ax1.axhline(y=15, color='orange', linestyle='--', alpha=0.7, label='Acceptable (15 TPS)')
    
    ax1.set_xlabel('Time (minutes)')
    ax1.set_ylabel('TPS (Ticks Per Second)')
    ax1.set_title('🎮 Luanti Server Performance - TPS Over Time', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 21)
    
    # Performance distribution histogram
    ax2 = plt.subplot(4, 3, 3)
    ax2.hist(tick_df['tps'], bins=50, color='skyblue', alpha=0.7, edgecolor='black')
    ax2.axvline(x=20, color='green', linestyle='--', linewidth=2, label='Target (20 TPS)')
    ax2.axvline(x=tick_df['tps'].mean(), color='red', linestyle='-', linewidth=2, 
                label=f'Average ({tick_df["tps"].mean():.1f} TPS)')
    ax2.set_xlabel('TPS')
    ax2.set_ylabel('Frequency')
    ax2.set_title('📊 TPS Distribution')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

# System metrics plots
if not cpu_df.empty:
    # CPU utilization plot
    ax3 = plt.subplot(4, 3, (4, 5))
    custom_params = {"axes.spines.right": False, "axes.spines.top": False}
    sns.set_theme(style="ticks", rc=custom_params)
    
    sns.lineplot(data=cpu_df, x="timestamp_m", y="cpu_util", hue="node", marker="o", markersize=2, ax=ax3)
    ax3.grid(axis="y", alpha=0.7)
    ax3.set_ylim(bottom=0)
    ax3.set_ylabel("CPU Utilization [%]")
    ax3.set_xlabel("Time [minutes]")
    ax3.set_title("💻 CPU Utilization During Benchmark")

if not mem_df.empty:
    # Memory utilization plot
    ax4 = plt.subplot(4, 3, 6)
    sns.lineplot(data=mem_df, x="timestamp_m", y="used_percent", hue="node", marker="o", markersize=2, ax=ax4)
    ax4.grid(axis="y", alpha=0.7)
    ax4.set_ylim(bottom=0, top=100)
    ax4.set_ylabel("Memory Utilization [%]")
    ax4.set_xlabel("Time [minutes]")
    ax4.set_title("💾 Memory Utilization During Benchmark")

if not net_df.empty:
    # Network throughput plot
    ax5 = plt.subplot(4, 3, (7, 8))
    
    # Calculate total throughput per node
    net_summary = net_df.groupby(['node', 'timestamp_m']).agg({
        'bytes_recv_rate': 'sum',
        'bytes_sent_rate': 'sum'
    }).reset_index()
    
    net_summary['total_rate_mbps'] = (net_summary['bytes_recv_rate'] + net_summary['bytes_sent_rate']) / (1024**2)
    
    sns.lineplot(data=net_summary, x="timestamp_m", y="total_rate_mbps", hue="node", marker="o", markersize=2, ax=ax5)
    ax5.grid(axis="y", alpha=0.7)
    ax5.set_ylim(bottom=0)
    ax5.set_ylabel("Network Throughput [MB/s]")
    ax5.set_xlabel("Time [minutes]")
    ax5.set_title("🌐 Network Throughput During Benchmark")

# Bot connection analysis
if 'player_df' in locals() and not player_df.empty and 'event_type' in player_df.columns:
    ax6 = plt.subplot(4, 3, 9)
    
    join_events = player_df[player_df['event_type'] == 'join'].copy() if 'join' in player_df['event_type'].values else pd.DataFrame()
    leave_events = player_df[player_df['event_type'] == 'leave'].copy() if 'leave' in player_df['event_type'].values else pd.DataFrame()
    
    if not join_events.empty:
        all_events = []
        for _, row in join_events.iterrows():
            all_events.append((row['timestamp'], 1, row['player_name']))
        for _, row in leave_events.iterrows():
            all_events.append((row['timestamp'], -1, row['player_name']))
        
        all_events.sort(key=lambda x: x[0])
        
        timestamps = []
        player_counts = []
        current_players = set()
        
        for timestamp, change, player in all_events:
            if change == 1:
                current_players.add(player)
            else:
                current_players.discard(player)
            timestamps.append(timestamp)
            player_counts.append(len(current_players))
        
        if timestamps:
            start_time = min(timestamps)
            time_minutes = [(t - start_time).total_seconds() / 60 for t in timestamps]
            ax6.plot(time_minutes, player_counts, 'b-', linewidth=2, alpha=0.7)
            ax6.fill_between(time_minutes, player_counts, alpha=0.3)
            ax6.axhline(y=200, color='red', linestyle='--', alpha=0.7, label='Target (200 bots)')
            ax6.set_ylabel('Concurrent Players')
            ax6.set_xlabel('Time (minutes)')
            ax6.set_title('🤖 Bot Connections Over Time')
            ax6.legend()
            ax6.grid(True, alpha=0.3)

# Performance summary text box
ax7 = plt.subplot(4, 3, (10, 12))
ax7.axis('off')

summary_text = "📋 COMPREHENSIVE BENCHMARK SUMMARY\n\n"

if 'tick_df' in locals() and not tick_df.empty:
    total_duration = (tick_df['timestamp'].max() - tick_df['timestamp'].min()).total_seconds()
    avg_tps = tick_df['tps'].mean()
    min_tps = tick_df['tps'].min()
    max_tps = tick_df['tps'].max()
    excellent_pct = (tick_df['tps'] >= 19.0).mean() * 100
    
    summary_text += f"⏰ Duration: {total_duration/60:.1f} minutes\n"
    summary_text += f"📊 Average TPS: {avg_tps:.2f}\n"
    summary_text += f"📈 TPS Range: {min_tps:.1f} - {max_tps:.1f}\n"
    summary_text += f"🎖️ Excellent Performance: {excellent_pct:.1f}%\n\n"

if not cpu_df.empty:
    overall_avg_cpu = cpu_df['cpu_util'].mean()
    overall_max_cpu = cpu_df['cpu_util'].max()
    summary_text += f"💻 CPU: {overall_avg_cpu:.1f}% avg, {overall_max_cpu:.1f}% peak\n"

if not mem_df.empty:
    overall_avg_mem = mem_df['used_percent'].mean()
    overall_max_mem = mem_df['used_percent'].max()
    summary_text += f"💾 Memory: {overall_avg_mem:.1f}% avg, {overall_max_mem:.1f}% peak\n"

if not net_df.empty:
    total_traffic_gb = net_df['total_bytes'].sum() / (1024**3)
    summary_text += f"🌐 Network: {total_traffic_gb:.2f}GB total traffic\n"

# Overall assessment
if 'tick_df' in locals() and not tick_df.empty:
    if avg_tps >= 18.0:
        summary_text += f"\n🎯 OVERALL: ✅ EXCELLENT PERFORMANCE"
    elif avg_tps >= 15.0:
        summary_text += f"\n🎯 OVERALL: ✅ GOOD PERFORMANCE"
    else:
        summary_text += f"\n🎯 OVERALL: ⚠️ NEEDS IMPROVEMENT"

ax7.text(0.1, 0.9, summary_text, transform=ax7.transAxes, fontsize=11,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.tight_layout()
plt.show()

# === REQUIRED SUMMARY PRINTS ===
print("\n🧾 REQUIRED METRICS SUMMARY")
print("=" * 30)

# 1) Total tick records
total_tick_records = len(tick_df)

# 2) Max players online (at the same time)
# Priority: direct column if present; otherwise use reconstructed concurrency from join/leave.
# --- Max concurrent players (robust, event-based) ---
max_players_online_same_time = None

have_events = (
    'player_df' in locals() and
    not player_df.empty and
    {'timestamp', 'event_type', 'player_name'}.issubset(player_df.columns)
)

if have_events:
    events = (
        player_df
        .loc[player_df['event_type'].isin(['join', 'leave']), ['timestamp', 'event_type', 'player_name']]
        .dropna()
        .drop_duplicates(subset=['timestamp', 'event_type', 'player_name'])  # remove exact dupes
        .copy()
    )

    # Ensure joins are processed before leaves at the same timestamp
    events['__order'] = events['event_type'].map({'join': 0, 'leave': 1})
    events = events.sort_values(['timestamp', '__order']).drop(columns='__order')

    current = set()
    max_concurrent = 0

    for ts, et, name in events[['timestamp', 'event_type', 'player_name']].itertuples(index=False):
        if et == 'join':
            # ignore duplicate join if already present
            if name not in current:
                current.add(name)
                if len(current) > max_concurrent:
                    max_concurrent = len(current)
        else:  # leave
            # ignore stray leave if not present
            if name in current:
                current.remove(name)

    max_players_online_same_time = int(max_concurrent)

# Fallbacks only if we don't have event data
elif 'players_online' in tick_df.columns:
    # Beware: some logs store cumulative counts here; use only as a fallback
    max_players_online_same_time = int(tick_df['players_online'].max())
elif 'players' in tick_df.columns:
    max_players_online_same_time = int(tick_df['players'].max())

# Print
if max_players_online_same_time is not None:
    print(f"- max players online (at the same time): {max_players_online_same_time}")
else:
    print(f"- max players online (at the same time): ❌ Data not available")


# 3) Average & 4) Peak tick duration (ms)
average_tick_duration_ms = float(tick_df['tick_duration'].mean())
peak_tick_duration_ms = float(tick_df['tick_duration'].max())

# 5) Average TPS
average_tps = float(tick_df['tps'].mean())

# 6) Lag events (count) — unify as ticks where duration>50ms OR TPS<20
lag_events_count = int(((tick_df['tick_duration'] > 50.0) | (tick_df['tps'] < 20.0)).sum())

# 7) Worst lag (in ms) — same as peak tick duration
worst_lag_ms = peak_tick_duration_ms

# Print neatly
print(f"- total tick records: {total_tick_records:,}")
if max_players_online_same_time is not None:
    print(f"- max players online (at the same time): {max_players_online_same_time}")
else:
    print(f"- max players online (at the same time): ❌ Data not available")
print(f"- average tick duration (ms): {average_tick_duration_ms:.2f}")
print(f"- peak tick duration (ms): {peak_tick_duration_ms:.2f}")
print(f"- average TPS: {average_tps:.2f}")
print(f"- lag events (count): {lag_events_count:,}")
print(f"- worst lag (ms): {worst_lag_ms:.2f}")

In [ ]:
# 🕐 SMART TEST PERIOD IDENTIFICATION & TIME-ALIGNED PREPROCESSING
print("🕐 SMART TEST PERIOD IDENTIFICATION & TIME-ALIGNED PREPROCESSING")
print("=" * 70)
print("🎯 Finding actual experiment timestamps to filter system metrics")

import pandas as pd
import subprocess
import os
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timezone

# === STEP 1: IDENTIFY ACTUAL TEST PERIOD FROM TICK DATA ===
print("\n📊 STEP 1: Identifying actual test period from tick data...")

test_start_time = None
test_end_time = None  
test_duration_minutes = None
test_start_timestamp = None
test_end_timestamp = None

# First check if we have tick data available to determine test period
if 'tick_df' in locals() and not tick_df.empty and 'timestamp_s' in tick_df.columns:
    print("   ✅ Using tick data to determine test period")
    
    # Get actual test boundaries from tick data (this is game server time)
    test_start_timestamp = tick_df['timestamp_s'].min()
    test_end_timestamp = tick_df['timestamp_s'].max()
    test_duration_minutes = (test_end_timestamp - test_start_timestamp) / 60
    
    print(f"   📅 Test period (from tick data):")
    print(f"      Start: {test_start_timestamp:.3f}")
    print(f"      End: {test_end_timestamp:.3f}")
    print(f"      Duration: {test_duration_minutes:.2f} minutes")
    
    # Convert to human readable times for verification
    try:
        start_dt = datetime.fromtimestamp(test_start_timestamp)
        end_dt = datetime.fromtimestamp(test_end_timestamp)
        print(f"      Human time: {start_dt.strftime('%H:%M:%S')} to {end_dt.strftime('%H:%M:%S')}")
    except:
        print(f"      (Timestamps appear to be relative, not absolute)")
    
    test_start_time = test_start_timestamp
    test_end_time = test_end_timestamp
else:
    print("   ❌ No tick data available - will include all system metrics")
    print("   ⚠️  Warning: System metrics may include pre-test startup data")

# === STEP 2: SMART SYSTEM METRICS TIME FILTERING ===
print(f"\n🔧 STEP 2: Smart time filtering for system metrics...")

def smart_timestamp_filter(timestamp, test_start, test_end, verbose=False):

    if test_start is None or test_end is None:
        return True  # Include everything if we can't determine test period
    
    # Case 1: Unix timestamp (> 1000000000, roughly after year 2001)
    if timestamp > 1000000000:
        # This is likely a Unix timestamp from Telegraf startup
        # Include some buffer before test start for context
        buffer_seconds = 60  # 1 minute buffer before test
        return timestamp >= (test_start - buffer_seconds)
    
    # Case 2: Relative timestamp (like tick data uses)
    else:
        # Relative timestamps should be within our test window
        return test_start <= timestamp <= test_end

# === STEP 3: FIND AND ANALYZE SYSTEM METRICS FILES ===
print(f"\n📁 STEP 3: Analyzing system metrics files...")

system_files = []
yardstick_output_path = '/var/scratch/aco237/yardstick/luanti_output'

if os.path.exists(yardstick_output_path):
    for item in os.listdir(yardstick_output_path):
        item_path = os.path.join(yardstick_output_path, item)
        if os.path.isdir(item_path) and item.startswith('node'):
            for subitem in os.listdir(item_path):
                subitem_path = os.path.join(item_path, subitem)
                if os.path.isdir(subitem_path) and subitem.startswith('telegraf-'):
                    metrics_file = os.path.join(subitem_path, f'metrics-{item}.csv')
                    if os.path.exists(metrics_file):
                        system_files.append(metrics_file)

print(f"📊 FOUND SYSTEM METRICS FILES:")
total_size = 0
for f in system_files:
    if os.path.exists(f):
        size = os.path.getsize(f)
        total_size += size
        print(f"   ✅ {f} ({size:,} bytes)")
    else:
        print(f"   ❌ {f} (not found)")

if system_files:
    print(f"📈 Total system metrics data: {total_size:,} bytes ({total_size/(1024**2):.1f} MB)")
else:
    print(f"❌ No system metrics files found!")

# === STEP 4: ANALYZE TIMESTAMP RANGES IN SYSTEM METRICS ===
print(f"\n⏰ STEP 4: Analyzing timestamp ranges in system metrics...")

if system_files and test_start_time is not None:
    for file_path in system_files[:1]:  # Just check first file for timing analysis
        node_name = os.path.basename(file_path).replace('metrics-', '').replace('.csv', '')
        print(f"   🔍 Analyzing timestamps in {node_name}...")
        
        # Get first and last few timestamps to understand the time range
        head_cmd = f"head -n 5 {file_path} | cut -d',' -f1"
        tail_cmd = f"tail -n 5 {file_path} | cut -d',' -f1"
        
        try:
            head_result = subprocess.run(head_cmd, shell=True, capture_output=True, text=True)
            tail_result = subprocess.run(tail_cmd, shell=True, capture_output=True, text=True)
            
            if head_result.returncode == 0 and tail_result.returncode == 0:
                head_timestamps = [int(line.strip()) for line in head_result.stdout.strip().split('\n') if line.strip().isdigit()]
                tail_timestamps = [int(line.strip()) for line in tail_result.stdout.strip().split('\n') if line.strip().isdigit()]
                
                if head_timestamps and tail_timestamps:
                    earliest = min(head_timestamps)
                    latest = max(tail_timestamps)
                    total_duration = (latest - earliest) / 60  # minutes
                    
                    print(f"      📅 System metrics time range:")
                    print(f"         Earliest: {earliest}")
                    print(f"         Latest: {latest}")
                    print(f"         Total duration: {total_duration:.1f} minutes")
                    
                    # Check if this overlaps with our test period
                    if earliest > 1000000000:  # Unix timestamp
                        print(f"      🕐 System metrics use Unix timestamps")
                        if test_start_time < 1000000000:
                            print(f"      ⚠️  Test data uses relative timestamps - conversion needed")
                    else:
                        print(f"      🕐 System metrics use relative timestamps")
                    
                    # Calculate how much data will be filtered out
                    if test_start_time is not None and test_end_time is not None:
                        test_duration_minutes = (test_end_time - test_start_time) / 60
                        filter_ratio = test_duration_minutes / total_duration
                        print(f"      🎯 Expected data after filtering: {filter_ratio*100:.1f}% ({test_duration_minutes:.1f} of {total_duration:.1f} minutes)")
                    
        except Exception as e:
            print(f"      ❌ Error analyzing timestamps: {e}")

print(f"\n✅ TEST PERIOD IDENTIFICATION COMPLETE!")
if test_start_time is not None:
    print(f"🎯 Identified test period: {test_duration_minutes:.2f} minutes")
    print(f"🔧 System metrics will be filtered to actual experiment time")
else:
    print(f"⚠️  Could not identify test period - will include all system metrics")
    print(f"📝 This may include Telegraf startup and other non-test data")

## Clean up nodes

In [ ]:
yardstick_benchmark.clean(nodes)
das.release(nodes)

## ✅ **MYSTERY SOLVED: Luanti Log File Separation**

### **Why You Only See Warnings:**

Luanti server uses **two separate log files** with different purposes:

1. **`server.log`** - Contains:
   - WARNING messages (like high tick duration)
   - ERROR messages  
   - System-level events
   - Performance alerts

2. **`startup.log`** - Contains:
   - ACTION messages (player joins/leaves)
   - Game events
   - Server initialization
   - Player interactions

### **The Numbers:**
- **server.log**: 0 player join messages (only warnings/errors)
- **startup.log**: 261 player join messages ✅

### **This is Normal Behavior!**
This log separation is intentional - Luanti separates operational warnings from game events. You've been looking at the right data all along, just in the wrong file!

### **To See Player Activity:**
- Use `startup.log` for player joins/leaves
- Use `server.log` for performance monitoring
- Both files are important for complete analysis

In [ ]:
# Updated download script to get BOTH log files
def download_complete_server_logs():
    """Download both server.log and startup.log for complete analysis"""
    
    print("📥 COMPLETE SERVER LOG DOWNLOAD")
    print("=" * 35)
    
    import subprocess
    import os
    from pathlib import Path
    from datetime import datetime
    
    username = os.getlogin()
    download_dir = Path(f"/var/scratch/aco237/server_logs_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    download_dir.mkdir(parents=True, exist_ok=True)
    
    # Both important log files
    log_files_to_download = [
        'server.log',    # Performance warnings, errors
        'startup.log',   # Player joins, leaves, actions
    ]
    
    total_downloaded = 0
    
    for node in nodes:
        print(f"\n🔍 Downloading from: {node.host}")
        
        try:
            # Find server directory
            find_cmd = f'ssh {node.host} "find /local/{username}/yardstick/*/luanti_server-* -type d 2>/dev/null | head -1"'
            server_dir_result = subprocess.run(find_cmd, shell=True, capture_output=True, text=True)
            
            if server_dir_result.returncode == 0 and server_dir_result.stdout.strip():
                server_dir = server_dir_result.stdout.strip()
                server_name = Path(server_dir).name
                
                for log_file in log_files_to_download:
                    remote_path = f"{server_dir}/logs/{log_file}"
                    local_path = download_dir / f"{node.host}_{log_file}"
                    
                    # Check file size first
                    size_cmd = f'ssh {node.host} "ls -l {remote_path} 2>/dev/null | awk \'{{print \\$5}}\' || echo 0"'
                    size_result = subprocess.run(size_cmd, shell=True, capture_output=True, text=True)
                    
                    try:
                        file_size = int(size_result.stdout.strip())
                        if file_size > 0:
                            # Download the file
                            scp_cmd = f"scp {username}@{node.host}:{remote_path} {local_path}"
                            scp_res = subprocess.run(scp_cmd, shell=True, capture_output=True, text=True)
                            
                            if scp_res.returncode == 0:
                                print(f"  ✅ {log_file}: {file_size:,} bytes downloaded")
                                total_downloaded += 1
                            else:
                                print(f"  ❌ {log_file}: Download failed")
                        else:
                            print(f"  ⚠️  {log_file}: Empty or missing")
                    except ValueError:
                        print(f"  ❌ {log_file}: Could not check size")
            else:
                print(f"  ❌ No server directory found")
                
        except Exception as e:
            print(f"  ❌ Error accessing {node.host}: {e}")
    
    print(f"\n🎯 DOWNLOAD COMPLETE")
    print(f"📊 Total files downloaded: {total_downloaded}")
    print(f"📁 Saved to: {download_dir}")
    
    # Analyze the downloaded logs
    if total_downloaded > 0:
        print(f"\n📋 Log Analysis Summary:")
        print("-" * 25)
        
        for log_file in download_dir.glob("*"):
            print(f"\n📄 {log_file.name}:")
            
            try:
                with open(log_file, 'r') as f:
                    content = f.read()
                    lines = content.split('\n')
                
                # Count different types of events
                if 'server.log' in log_file.name:
                    warnings = len([line for line in lines if 'WARNING' in line])
                    errors = len([line for line in lines if 'ERROR' in line])
                    yardstick_warnings = len([line for line in lines if 'YARDSTICK: High tick duration' in line])
                    
                    print(f"  ⚠️  Total warnings: {warnings:,}")
                    print(f"  ❌ Total errors: {errors:,}")
                    print(f"  🐌 High tick warnings: {yardstick_warnings:,}")
                    
                elif 'startup.log' in log_file.name:
                    joins = len([line for line in lines if 'joins game' in line])
                    leaves = len([line for line in lines if 'leaves game' in line])
                    actions = len([line for line in lines if 'ACTION[' in line])
                    
                    print(f"  🚪 Player joins: {joins:,}")
                    print(f"  🚪 Player leaves: {leaves:,}")
                    print(f"  🎮 Total actions: {actions:,}")
                
                # Show time range
                import re
                timestamp_pattern = r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})'
                first_match = re.search(timestamp_pattern, content)
                
                last_lines = lines[-50:]  # Check last 50 lines for timestamp
                last_match = None
                for line in reversed(last_lines):
                    last_match = re.search(timestamp_pattern, line)
                    if last_match:
                        break
                
                if first_match and last_match:
                    print(f"  ⏰ Time range: {first_match.group(1)} to {last_match.group(1)}")
                
            except Exception as e:
                print(f"  ❌ Error analyzing {log_file.name}: {e}")
    
    return download_dir

# Download both log files
complete_logs_dir = download_complete_server_logs()

## Packet Quota System Analysis

### Understanding "Packet quota used up" Warnings

The server logs show warnings like:
```
WARNING[ConnectionSend]: con(4/1) Packet quota used up for peer_id=64003, was 13 pkts
```

**Key Insight**: The `max_packets_per_iteration = 2048` setting controls the **total** packet budget per server iteration, but this gets **divided among all connected players**.

### How Packet Quota Works

1. **Total Budget**: `max_packets_per_iteration = 2048` packets per server iteration
2. **Per-Player Quota**: `packet_quota_per_peer = total_budget / number_of_active_peers`
3. **Formula**: `peer_packet_quota = MYMAX(1, m_iteration_packets_available / numpeers)`

### Calculation for Our Benchmark

- **max_packets_per_iteration**: 2048 (configured, 2x default of 1024)
- **Peak concurrent players**: ~171 bots
- **Quota per peer**: 2048 ÷ 171 ≈ **12-13 packets per peer per iteration**

This explains why the logs show "was 13 pkts" - each bot was allocated ~13 packets per server iteration, and some bots used up their entire quota.

### Implications

- The `max_packets_per_iteration` setting **was working correctly**
- The limitation was **per-peer quota distribution**, not total capacity
- With more players, each gets fewer packets per iteration
- This is a **load balancing mechanism** to prevent any single player from monopolizing network resources

### Source Code Reference

From `luanti/src/network/mtp/threads.cpp`:
```cpp
const auto &calculate_quota = [&] () -> u32 {
    u32 numpeers = m_connection->getActiveCount();
    if (numpeers > 0)
        return MYMAX(1, m_iteration_packets_avaialble / numpeers);
    return m_iteration_packets_avaialble;
};
```

In [ ]:
# 🎯 TIME-FILTERED SYSTEM METRICS EXTRACTION
print("🎯 TIME-FILTERED SYSTEM METRICS EXTRACTION")
print("=" * 50)
print("⏰ Processing system metrics using smart time filtering")

# Reuse the system files found in preprocessing
if 'system_files' not in locals():
    system_files = []
    yardstick_output_path = '/var/scratch/aco237/yardstick/luanti_output'
    
    if os.path.exists(yardstick_output_path):
        for item in os.listdir(yardstick_output_path):
            item_path = os.path.join(yardstick_output_path, item)
            if os.path.isdir(item_path) and item.startswith('node'):
                for subitem in os.listdir(item_path):
                    subitem_path = os.path.join(item_path, subitem)
                    if os.path.isdir(subitem_path) and subitem.startswith('telegraf-'):
                        metrics_file = os.path.join(subitem_path, f'metrics-{item}.csv')
                        if os.path.exists(metrics_file):
                            system_files.append(metrics_file)

print("📊 PROCESSING SYSTEM METRICS FILES:")
for f in system_files:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f"   ✅ {f} ({size:,} bytes)")
    else:
        print(f"   ❌ {f} (not found)")

# Initialize data containers
cpu_data_nodes = []
mem_data_nodes = []
net_data_nodes = []

# Extract all system metrics efficiently
for file_path in system_files:
    if os.path.exists(file_path):
        node_name = os.path.basename(file_path).replace('metrics-', '').replace('.csv', '')
        print(f"\n🖥️ Processing system data from {node_name}...")
        
        try:
            # === CPU METRICS EXTRACTION ===
            cpu_cmd = f"grep '^[0-9]*,cpu,' {file_path}"
            result = subprocess.run(cpu_cmd, shell=True, capture_output=True, text=True)
            
            if result.returncode == 0 and result.stdout.strip():
                lines = result.stdout.strip().split('\n')
                print(f"   📈 Total CPU lines found: {len(lines)}")
                
                cpu_records = []
                filtered_cpu_count = 0
                skipped_cpu_count = 0
                
                for line in lines:
                    parts = line.split(',')
                    if len(parts) >= 17:
                        try:
                            timestamp = int(parts[0])
                            
                            # Apply smart time filtering for CPU metrics
                            if not smart_timestamp_filter(timestamp, test_start_time, test_end_time):
                                skipped_cpu_count += 1
                                continue
                            
                            filtered_cpu_count += 1
                            cpu_id = parts[2]
                            cpu_name = parts[3]
                            
                            # CPU time fields - based on the telegraf CPU plugin format
                            time_guest = float(parts[5]) if parts[5] else 0
                            time_guest_nice = float(parts[6]) if parts[6] else 0
                            time_idle = float(parts[9]) if parts[9] else 0
                            time_iowait = float(parts[10]) if parts[10] else 0
                            time_irq = float(parts[11]) if parts[11] else 0
                            time_nice = float(parts[12]) if parts[12] else 0
                            time_softirq = float(parts[13]) if parts[13] else 0
                            time_steal = float(parts[14]) if parts[14] else 0
                            time_system = float(parts[15]) if parts[15] else 0
                            time_user = float(parts[16]) if parts[16] else 0
                            
                            # Calculate utilization
                            time_active = time_user + time_system + time_nice + time_irq + time_softirq + time_steal + time_guest + time_guest_nice
                            time_total = time_active + time_idle + time_iowait
                            
                            if time_total > 0:
                                cpu_util = (time_active / time_total) * 100
                                
                                cpu_records.append({
                                    'timestamp': timestamp,
                                    'node': node_name,
                                    'cpu_id': cpu_id,
                                    'cpu_name': cpu_name,
                                    'cpu_util': cpu_util,
                                    'time_user': time_user,
                                    'time_system': time_system,
                                    'time_idle': time_idle,
                                    'time_iowait': time_iowait
                                })
                        except (ValueError, IndexError):
                            continue
                
                print(f"   🎯 CPU filtering results: kept {filtered_cpu_count}, skipped {skipped_cpu_count}")
                
                if cpu_records:
                    node_cpu_df = pd.DataFrame(cpu_records)
                    
                    # Calculate overall CPU utilization per timestamp (average across cores)
                    cpu_avg = node_cpu_df.groupby(['timestamp', 'node']).agg({
                        'cpu_util': 'mean',
                        'time_user': 'mean',
                        'time_system': 'mean',
                        'time_idle': 'mean',
                        'time_iowait': 'mean'
                    }).reset_index()
                    
                    # Normalize timestamps relative to test start
                    if test_start_time is not None:
                        cpu_avg['timestamp_norm'] = cpu_avg['timestamp'] - test_start_time
                        cpu_avg['timestamp_m'] = cpu_avg['timestamp_norm'] / 60
                    else:
                        cpu_avg['timestamp_norm'] = cpu_avg['timestamp'] - cpu_avg['timestamp'].min()
                        cpu_avg['timestamp_m'] = cpu_avg['timestamp_norm'] / 60
                    
                    cpu_data_nodes.append(cpu_avg)
                    
                    avg_util = cpu_avg['cpu_util'].mean()
                    max_util = cpu_avg['cpu_util'].max()
                    print(f"   📊 {node_name}: CPU Avg={avg_util:.1f}%, Max={max_util:.1f}%")
            
            # === MEMORY METRICS EXTRACTION ===
            mem_cmd = f"grep '^[0-9]*,mem,' {file_path}"
            mem_result = subprocess.run(mem_cmd, shell=True, capture_output=True, text=True)
            
            if mem_result.returncode == 0 and mem_result.stdout.strip():
                lines = mem_result.stdout.strip().split('\n')
                print(f"   💾 Total memory lines found: {len(lines)}")
                
                mem_records = []
                filtered_mem_count = 0
                skipped_mem_count = 0
                
                for line in lines:
                    parts = line.split(',')
                    if len(parts) >= 10:
                        try:
                            timestamp = int(parts[0])
                            
                            # Apply smart time filtering for memory metrics
                            if not smart_timestamp_filter(timestamp, test_start_time, test_end_time):
                                skipped_mem_count += 1
                                continue
                            
                            filtered_mem_count += 1
                            
                            # Memory fields based on telegraf mem plugin
                            available = float(parts[3]) if parts[3] else 0
                            total = float(parts[4]) if parts[4] else 0
                            used_percent = float(parts[5]) if parts[5] else 0
                            
                            if total > 0 and 0 <= used_percent <= 100:
                                used = total * (used_percent / 100)
                                
                                mem_records.append({
                                    'timestamp': timestamp,
                                    'node': node_name,
                                    'total_gb': total / (1024**3),
                                    'used_gb': used / (1024**3),
                                    'available_gb': available / (1024**3),
                                    'used_percent': used_percent
                                })
                        except (ValueError, IndexError):
                            continue
                
                print(f"   🎯 Memory filtering results: kept {filtered_mem_count}, skipped {skipped_mem_count}")
                
                if mem_records:
                    node_mem_df = pd.DataFrame(mem_records)
                    
                    # Normalize timestamps relative to test start
                    if test_start_time is not None:
                        node_mem_df['timestamp_norm'] = node_mem_df['timestamp'] - test_start_time
                        node_mem_df['timestamp_m'] = node_mem_df['timestamp_norm'] / 60
                    else:
                        node_mem_df['timestamp_norm'] = node_mem_df['timestamp'] - node_mem_df['timestamp'].min()
                        node_mem_df['timestamp_m'] = node_mem_df['timestamp_norm'] / 60
                    
                    mem_data_nodes.append(node_mem_df)
                    
                    avg_mem = node_mem_df['used_percent'].mean()
                    max_mem = node_mem_df['used_percent'].max()
                    avg_used_gb = node_mem_df['used_gb'].mean()
                    print(f"   📊 {node_name}: Memory Avg={avg_mem:.1f}%, Max={max_mem:.1f}%, Avg Used={avg_used_gb:.1f}GB")
            
            # === NETWORK METRICS EXTRACTION ===
            net_cmd = f"grep '^[0-9]*,net,' {file_path}"
            net_result = subprocess.run(net_cmd, shell=True, capture_output=True, text=True)
            
            if net_result.returncode == 0 and net_result.stdout.strip():
                lines = net_result.stdout.strip().split('\n')
                print(f"   🌐 Total network lines found: {len(lines)}")
                
                net_records = []
                filtered_net_count = 0
                skipped_net_count = 0
                
                for line in lines:
                    parts = line.split(',')
                    if len(parts) >= 10:
                        try:
                            timestamp = int(parts[0])
                            
                            # Apply smart time filtering for network metrics
                            if not smart_timestamp_filter(timestamp, test_start_time, test_end_time):
                                skipped_net_count += 1
                                continue
                            
                            filtered_net_count += 1
                            interface = parts[3]
                            
                            # Only process main network interfaces (skip loopback, etc.)
                            if any(iface in interface for iface in ['eth', 'ens', 'enp']) and interface != 'lo':
                                bytes_recv = float(parts[4]) if parts[4] else 0
                                bytes_sent = float(parts[5]) if parts[5] else 0
                                packets_recv = float(parts[8]) if len(parts) > 8 and parts[8] else 0
                                packets_sent = float(parts[9]) if len(parts) > 9 and parts[9] else 0
                                
                                net_records.append({
                                    'timestamp': timestamp,
                                    'node': node_name,
                                    'interface': interface,
                                    'bytes_recv': bytes_recv,
                                    'bytes_sent': bytes_sent,
                                    'packets_recv': packets_recv,
                                    'packets_sent': packets_sent,
                                    'total_bytes': bytes_recv + bytes_sent,
                                    'total_packets': packets_recv + packets_sent
                                })
                        except (ValueError, IndexError):
                            continue
                
                print(f"   🎯 Network filtering results: kept {filtered_net_count}, skipped {skipped_net_count}")
                
                if net_records:
                    node_net_df = pd.DataFrame(net_records)
                    
                    # Normalize timestamps relative to test start  
                    if test_start_time is not None:
                        node_net_df['timestamp_norm'] = node_net_df['timestamp'] - test_start_time
                        node_net_df['timestamp_m'] = node_net_df['timestamp_norm'] / 60
                    else:
                        node_net_df['timestamp_norm'] = node_net_df['timestamp'] - node_net_df['timestamp'].min()
                        node_net_df['timestamp_m'] = node_net_df['timestamp_norm'] / 60
                    
                    # Calculate throughput (bytes per second) for rate-based metrics
                    if len(node_net_df) > 1:
                        node_net_df = node_net_df.sort_values(['interface', 'timestamp'])
                        for interface in node_net_df['interface'].unique():
                            mask = node_net_df['interface'] == interface
                            interface_data = node_net_df[mask].copy()
                            if len(interface_data) > 1:
                                interface_data['bytes_recv_rate'] = interface_data['bytes_recv'].diff() / interface_data['timestamp'].diff()
                                interface_data['bytes_sent_rate'] = interface_data['bytes_sent'].diff() / interface_data['timestamp'].diff()
                                # Update the main dataframe
                                node_net_df.loc[mask, 'bytes_recv_rate'] = interface_data['bytes_recv_rate']
                                node_net_df.loc[mask, 'bytes_sent_rate'] = interface_data['bytes_sent_rate']
                        
                        node_net_df = node_net_df.fillna(0)
                        # Remove negative rates (can happen with counter resets)
                        node_net_df['bytes_recv_rate'] = node_net_df['bytes_recv_rate'].clip(lower=0)
                        node_net_df['bytes_sent_rate'] = node_net_df['bytes_sent_rate'].clip(lower=0)
                    
                    net_data_nodes.append(node_net_df)
                    
                    # Calculate total traffic
                    if 'bytes_recv_rate' in node_net_df.columns:
                        avg_recv_rate = node_net_df['bytes_recv_rate'].mean() / (1024**2)  # MB/s
                        avg_sent_rate = node_net_df['bytes_sent_rate'].mean() / (1024**2)  # MB/s
                        print(f"   📊 {node_name}: Network Avg Recv={avg_recv_rate:.2f}MB/s, Avg Sent={avg_sent_rate:.2f}MB/s")
                    
        except Exception as e:
            print(f"   ❌ Error processing {file_path}: {e}")

# Combine all system metrics
cpu_df = pd.concat(cpu_data_nodes, ignore_index=True) if cpu_data_nodes else pd.DataFrame()
mem_df = pd.concat(mem_data_nodes, ignore_index=True) if mem_data_nodes else pd.DataFrame()
net_df = pd.concat(net_data_nodes, ignore_index=True) if net_data_nodes else pd.DataFrame()

print(f"\n📊 SYSTEM METRICS SUMMARY:")
print(f"   CPU: {len(cpu_df)} records from {len(cpu_data_nodes)} nodes")
print(f"   Memory: {len(mem_df)} records from {len(mem_data_nodes)} nodes")
print(f"   Network: {len(net_df)} records from {len(net_data_nodes)} nodes")

# === APPLICATION METRICS PREPROCESSING ===
print("\n🔧 PRE-PROCESSING APPLICATION METRICS")
print("=" * 40)

if 'tick_df' in locals() and not tick_df.empty:
    print("⚡ PROCESSING TICK METRICS...")
    
    # Clean and validate tick duration data
    tick_df = tick_df.copy()
    tick_df['tick_duration'] = pd.to_numeric(tick_df['tick_duration'], errors='coerce')
    tick_df = tick_df.dropna(subset=['tick_duration'])
    
    # Calculate TPS (Ticks Per Second)
    tick_df['tps'] = 1000.0 / tick_df['tick_duration']
    tick_df['tps'] = tick_df['tps'].clip(upper=20.0)
    
    # Add time-based analysis columns
    tick_df = tick_df.sort_values('timestamp')
    tick_df['time_from_start'] = (tick_df['timestamp'] - tick_df['timestamp'].min()).dt.total_seconds()
    
    # Calculate rolling averages
    tick_df['tps_1min_avg'] = tick_df['tps'].rolling(window=1200, min_periods=1).mean()
    tick_df['tps_5sec_avg'] = tick_df['tps'].rolling(window=100, min_periods=1).mean()
    
    # Performance statistics
    avg_tps = tick_df['tps'].mean()
    min_tps = tick_df['tps'].min()
    max_tps = tick_df['tps'].max()
    avg_duration = tick_df['tick_duration'].mean()
    
    print(f"  ✅ Processed {len(tick_df):,} tick records")
    print(f"  📊 Average TPS: {avg_tps:.2f} (target: 20.00)")
    print(f"  📊 TPS range: {min_tps:.2f} - {max_tps:.2f}")
    print(f"  ⏱️ Average tick duration: {avg_duration:.2f}ms (target: 50ms)")
    
    # Performance categories
    excellent_tps = (tick_df['tps'] >= 19.0).sum()
    good_tps = ((tick_df['tps'] >= 15.0) & (tick_df['tps'] < 19.0)).sum()
    poor_tps = (tick_df['tps'] < 15.0).sum()
    
    print(f"  🎯 Performance breakdown:")
    print(f"     Excellent (≥19 TPS): {excellent_tps:,} ({excellent_tps/len(tick_df)*100:.1f}%)")
    print(f"     Good (15-19 TPS): {good_tps:,} ({good_tps/len(tick_df)*100:.1f}%)")
    print(f"     Poor (<15 TPS): {poor_tps:,} ({poor_tps/len(tick_df)*100:.1f}%)")
    
    # === ADDITIONAL METRICS YOU REQUESTED ===
    print(f"\n📈 DETAILED PERFORMANCE METRICS:")
    print("=" * 35)
    
    # Total tick records
    total_tick_records = len(tick_df)
    print(f"📊 Total tick records: {total_tick_records:,}")
    
    # Max players online (if available)
    if 'players_online' in tick_df.columns:
        max_players_online = tick_df['players_online'].max()
        avg_players_online = tick_df['players_online'].mean()
        print(f"👥 Max players online (concurrent): {max_players_online}")
        print(f"👥 Average players online: {avg_players_online:.1f}")
    elif 'players' in tick_df.columns:
        max_players_online = tick_df['players'].max()
        avg_players_online = tick_df['players'].mean()
        print(f"👥 Max players online (concurrent): {max_players_online}")
        print(f"👥 Average players online: {avg_players_online:.1f}")
    else:
        print(f"👥 Max players online: ❌ Data not available")
    
    # Average and peak tick duration
    avg_tick_duration = tick_df['tick_duration'].mean()
    peak_tick_duration = tick_df['tick_duration'].max()
    print(f"⏱️  Average tick duration: {avg_tick_duration:.2f}ms")
    print(f"⏱️  Peak tick duration: {peak_tick_duration:.2f}ms")
    
    # Average TPS
    print(f"🎯 Average TPS: {avg_tps:.2f}")
    
    # Lag events count (define lag as tick duration > 50ms or TPS < 20)
    lag_threshold_ms = 50.0  # Consider >50ms as lag
    lag_threshold_tps = 20.0  # Consider <20 TPS as lag
    
    lag_events_duration = (tick_df['tick_duration'] > lag_threshold_ms).sum()
    lag_events_tps = (tick_df['tps'] < lag_threshold_tps).sum()
    
    print(f"🐌 Lag events (>50ms duration): {lag_events_duration:,}")
    print(f"🐌 Lag events (<20 TPS): {lag_events_tps:,}")
    
    # Severe lag events (>100ms or <10 TPS)
    severe_lag_duration = (tick_df['tick_duration'] > 100.0).sum()
    severe_lag_tps = (tick_df['tps'] < 10.0).sum()
    
    print(f"🔴 Severe lag events (>100ms): {severe_lag_duration:,}")
    print(f"🔴 Severe lag events (<10 TPS): {severe_lag_tps:,}")
    
    # Worst lag (highest tick duration)
    worst_lag_ms = tick_df['tick_duration'].max()
    worst_tps = tick_df['tps'].min()
    print(f"🐌 Worst lag spike: {worst_lag_ms:.2f}ms")
    print(f"🐌 Lowest TPS recorded: {worst_tps:.2f}")
    
    # Lag percentage of total time
    lag_percentage = (lag_events_duration / total_tick_records) * 100
    severe_lag_percentage = (severe_lag_duration / total_tick_records) * 100
    
    print(f"📊 Time with lag (>50ms): {lag_percentage:.2f}%")
    print(f"📊 Time with severe lag (>100ms): {severe_lag_percentage:.2f}%")
    
    # Benchmark duration
    if tick_df['timestamp'].dtype == 'datetime64[ns]':
        benchmark_duration = (tick_df['timestamp'].max() - tick_df['timestamp'].min()).total_seconds()
    else:
        benchmark_duration = tick_df['time_from_start'].max()
    
    print(f"⏰ Total benchmark duration: {benchmark_duration/60:.1f} minutes")
    
else:
    print("❌ No tick data available for processing")

# Process player metrics
if 'player_df' in locals() and not player_df.empty:
    print(f"\n👥 PROCESSING PLAYER METRICS...")
    
    player_df = player_df.copy()
    player_df = player_df.sort_values('timestamp')
    player_df['time_from_start'] = (player_df['timestamp'] - player_df['timestamp'].min()).dt.total_seconds()
    
    unique_players = player_df['player_name'].nunique()
    total_events = len(player_df)
    event_types = player_df['event_type'].value_counts()
    
    print(f"  ✅ Processed {total_events:,} player events")
    print(f"  👥 Unique players: {unique_players}")
    print(f"  📈 Event breakdown:")
    for event_type, count in event_types.head(5).items():
        print(f"     {event_type}: {count:,}")
else:
    print("\n⚠️ No player data available for processing")

# === COMPREHENSIVE VISUALIZATIONS ===
print("\n📈 CREATING COMPREHENSIVE VISUALIZATIONS")
print("=" * 45)

# Create a comprehensive dashboard with both application and system metrics
fig = plt.figure(figsize=(20, 16))

# Application metrics plots (if available)
if 'tick_df' in locals() and not tick_df.empty:
    # Main TPS over time plot
    ax1 = plt.subplot(4, 3, (1, 2))
    
    colors = []
    for tps in tick_df['tps']:
        if tps >= 19.0:
            colors.append('green')
        elif tps >= 15.0:
            colors.append('orange')
        else:
            colors.append('red')
    
    scatter = ax1.scatter(tick_df['time_from_start']/60, tick_df['tps'], 
                         c=colors, alpha=0.6, s=1)
    
    ax1.plot(tick_df['time_from_start']/60, tick_df['tps_1min_avg'], 
             color='blue', linewidth=2, label='1-min average')
    
    ax1.axhline(y=20, color='green', linestyle='--', alpha=0.7, label='Target (20 TPS)')
    ax1.axhline(y=15, color='orange', linestyle='--', alpha=0.7, label='Acceptable (15 TPS)')
    
    ax1.set_xlabel('Time (minutes)')
    ax1.set_ylabel('TPS (Ticks Per Second)')
    ax1.set_title('🎮 Luanti Server Performance - TPS Over Time', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 21)
    
    # Performance distribution histogram
    ax2 = plt.subplot(4, 3, 3)
    ax2.hist(tick_df['tps'], bins=50, color='skyblue', alpha=0.7, edgecolor='black')
    ax2.axvline(x=20, color='green', linestyle='--', linewidth=2, label='Target (20 TPS)')
    ax2.axvline(x=tick_df['tps'].mean(), color='red', linestyle='-', linewidth=2, 
                label=f'Average ({tick_df["tps"].mean():.1f} TPS)')
    ax2.set_xlabel('TPS')
    ax2.set_ylabel('Frequency')
    ax2.set_title('📊 TPS Distribution')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

# System metrics plots
if not cpu_df.empty:
    # CPU utilization plot
    ax3 = plt.subplot(4, 3, (4, 5))
    custom_params = {"axes.spines.right": False, "axes.spines.top": False}
    sns.set_theme(style="ticks", rc=custom_params)
    
    sns.lineplot(data=cpu_df, x="timestamp_m", y="cpu_util", hue="node", marker="o", markersize=2, ax=ax3)
    ax3.grid(axis="y", alpha=0.7)
    ax3.set_ylim(bottom=0)
    ax3.set_ylabel("CPU Utilization [%]")
    ax3.set_xlabel("Time [minutes]")
    ax3.set_title("💻 CPU Utilization During Benchmark")

if not mem_df.empty:
    # Memory utilization plot
    ax4 = plt.subplot(4, 3, 6)
    sns.lineplot(data=mem_df, x="timestamp_m", y="used_percent", hue="node", marker="o", markersize=2, ax=ax4)
    ax4.grid(axis="y", alpha=0.7)
    ax4.set_ylim(bottom=0, top=100)
    ax4.set_ylabel("Memory Utilization [%]")
    ax4.set_xlabel("Time [minutes]")
    ax4.set_title("💾 Memory Utilization During Benchmark")

if not net_df.empty and 'bytes_recv_rate' in net_df.columns:
    # Network throughput plot
    ax5 = plt.subplot(4, 3, (7, 8))
    
    # Calculate total throughput per node per minute
    net_summary = net_df.groupby(['node', 'timestamp_m']).agg({
        'bytes_recv_rate': 'sum',
        'bytes_sent_rate': 'sum'
    }).reset_index()
    
    net_summary['total_rate_mbps'] = (net_summary['bytes_recv_rate'] + net_summary['bytes_sent_rate']) / (1024**2)
    
    sns.lineplot(data=net_summary, x="timestamp_m", y="total_rate_mbps", hue="node", marker="o", markersize=2, ax=ax5)
    ax5.grid(axis="y", alpha=0.7)
    ax5.set_ylim(bottom=0)
    ax5.set_ylabel("Network Throughput [MB/s]")
    ax5.set_xlabel("Time [minutes]")
    ax5.set_title("🌐 Network Throughput During Benchmark")

# Bot connection analysis
if 'player_df' in locals() and not player_df.empty and 'event_type' in player_df.columns:
    ax6 = plt.subplot(4, 3, 9)
    
    join_events = player_df[player_df['event_type'] == 'join'].copy() if 'join' in player_df['event_type'].values else pd.DataFrame()
    leave_events = player_df[player_df['event_type'] == 'leave'].copy() if 'leave' in player_df['event_type'].values else pd.DataFrame()
    
    if not join_events.empty:
        all_events = []
        for _, row in join_events.iterrows():
            all_events.append((row['timestamp'], 1, row['player_name']))
        for _, row in leave_events.iterrows():
            all_events.append((row['timestamp'], -1, row['player_name']))
        
        all_events.sort(key=lambda x: x[0])
        
        timestamps = []
        player_counts = []
        current_players = set()
        
        for timestamp, change, player in all_events:
            if change == 1:
                current_players.add(player)
            else:
                current_players.discard(player)
            timestamps.append(timestamp)
            player_counts.append(len(current_players))
        
        if timestamps:
            start_time = min(timestamps)
            time_minutes = [(t - start_time).total_seconds() / 60 for t in timestamps]
            ax6.plot(time_minutes, player_counts, 'b-', linewidth=2, alpha=0.7)
            ax6.fill_between(time_minutes, player_counts, alpha=0.3)
            ax6.axhline(y=200, color='red', linestyle='--', alpha=0.7, label='Target (200 bots)')
            ax6.set_ylabel('Concurrent Players')
            ax6.set_xlabel('Time (minutes)')
            ax6.set_title('🤖 Bot Connections Over Time')
            ax6.legend()
            ax6.grid(True, alpha=0.3)

# Performance summary text box
ax7 = plt.subplot(4, 3, (10, 12))
ax7.axis('off')

summary_text = "📋 COMPREHENSIVE BENCHMARK SUMMARY\n\n"

if 'tick_df' in locals() and not tick_df.empty:
    total_duration = (tick_df['timestamp'].max() - tick_df['timestamp'].min()).total_seconds()
    avg_tps = tick_df['tps'].mean()
    min_tps = tick_df['tps'].min()
    max_tps = tick_df['tps'].max()
    excellent_pct = (tick_df['tps'] >= 19.0).mean() * 100
    
    summary_text += f"⏰ Duration: {total_duration/60:.1f} minutes\n"
    summary_text += f"📊 Average TPS: {avg_tps:.2f}\n"
    summary_text += f"📈 TPS Range: {min_tps:.1f} - {max_tps:.1f}\n"
    summary_text += f"🎖️ Excellent Performance: {excellent_pct:.1f}%\n\n"

if not cpu_df.empty:
    overall_avg_cpu = cpu_df['cpu_util'].mean()
    overall_max_cpu = cpu_df['cpu_util'].max()
    summary_text += f"💻 CPU: {overall_avg_cpu:.1f}% avg, {overall_max_cpu:.1f}% peak\n"

if not mem_df.empty:
    overall_avg_mem = mem_df['used_percent'].mean()
    overall_max_mem = mem_df['used_percent'].max()
    summary_text += f"💾 Memory: {overall_avg_mem:.1f}% avg, {overall_max_mem:.1f}% peak\n"

if not net_df.empty and 'bytes_recv_rate' in net_df.columns:
    avg_throughput = ((net_df['bytes_recv_rate'] + net_df['bytes_sent_rate']) / (1024**2)).mean()
    summary_text += f"🌐 Network: {avg_throughput:.2f}MB/s avg throughput\n"

# Overall assessment
if 'tick_df' in locals() and not tick_df.empty:
    if avg_tps >= 18.0:
        summary_text += f"\n🎯 OVERALL: ✅ EXCELLENT PERFORMANCE"
    elif avg_tps >= 15.0:
        summary_text += f"\n🎯 OVERALL: ✅ GOOD PERFORMANCE"
    else:
        summary_text += f"\n🎯 OVERALL: ⚠️ NEEDS IMPROVEMENT"

ax7.text(0.1, 0.9, summary_text, transform=ax7.transAxes, fontsize=11,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.tight_layout()
plt.show()

# === REQUIRED SUMMARY PRINTS ===
print("\n🧾 REQUIRED METRICS SUMMARY")
print("=" * 30)

if 'tick_df' in locals() and not tick_df.empty:
    # 1) Total tick records
    total_tick_records = len(tick_df)

    # 2) Max players online (at the same time)
    # Priority: direct column if present; otherwise use reconstructed concurrency from join/leave.
    # --- Max concurrent players (robust, event-based) ---
    max_players_online_same_time = None

    have_events = (
        'player_df' in locals() and
        not player_df.empty and
        {'timestamp', 'event_type', 'player_name'}.issubset(player_df.columns)
    )

    if have_events:
        events = (
            player_df
            .loc[player_df['event_type'].isin(['join', 'leave']), ['timestamp', 'event_type', 'player_name']]
            .dropna()
            .drop_duplicates(subset=['timestamp', 'event_type', 'player_name'])  # remove exact dupes
            .copy()
        )

        # Ensure joins are processed before leaves at the same timestamp
        events['__order'] = events['event_type'].map({'join': 0, 'leave': 1})
        events = events.sort_values(['timestamp', '__order']).drop(columns='__order')

        current = set()
        max_concurrent = 0

        for ts, et, name in events[['timestamp', 'event_type', 'player_name']].itertuples(index=False):
            if et == 'join':
                # ignore duplicate join if already present
                if name not in current:
                    current.add(name)
                    if len(current) > max_concurrent:
                        max_concurrent = len(current)
            else:  # leave
                # ignore stray leave if not present
                if name in current:
                    current.remove(name)

        max_players_online_same_time = int(max_concurrent)

    # Fallbacks only if we don't have event data
    elif 'players_online' in tick_df.columns:
        # Beware: some logs store cumulative counts here; use only as a fallback
        max_players_online_same_time = int(tick_df['players_online'].max())
    elif 'players' in tick_df.columns:
        max_players_online_same_time = int(tick_df['players'].max())

    # 3) Average & 4) Peak tick duration (ms)
    average_tick_duration_ms = float(tick_df['tick_duration'].mean())
    peak_tick_duration_ms = float(tick_df['tick_duration'].max())

    # 5) Average TPS
    average_tps = float(tick_df['tps'].mean())

    # 6) Lag events (count) — unify as ticks where duration>50ms OR TPS<20
    lag_events_count = int(((tick_df['tick_duration'] > 50.0) | (tick_df['tps'] < 20.0)).sum())

    # 7) Worst lag (in ms) — same as peak tick duration
    worst_lag_ms = peak_tick_duration_ms

    # Print neatly
    print(f"- total tick records: {total_tick_records:,}")
    if max_players_online_same_time is not None:
        print(f"- max players online (at the same time): {max_players_online_same_time}")
    else:
        print(f"- max players online (at the same time): ❌ Data not available")
    print(f"- average tick duration (ms): {average_tick_duration_ms:.2f}")
    print(f"- peak tick duration (ms): {peak_tick_duration_ms:.2f}")
    print(f"- average TPS: {average_tps:.2f}")
    print(f"- lag events (count): {lag_events_count:,}")
    print(f"- worst lag (ms): {worst_lag_ms:.2f}")

    # === SYSTEM METRICS SUMMARY ===
    print(f"\n💻 SYSTEM METRICS SUMMARY")
    print("=" * 25)
    
    if not cpu_df.empty:
        overall_avg_cpu = cpu_df['cpu_util'].mean()
        overall_max_cpu = cpu_df['cpu_util'].max()
        overall_min_cpu = cpu_df['cpu_util'].min()
        print(f"- CPU utilization: avg={overall_avg_cpu:.1f}%, peak={overall_max_cpu:.1f}%, min={overall_min_cpu:.1f}%")
    else:
        print(f"- CPU utilization: ❌ Data not available")

    if not mem_df.empty:
        overall_avg_mem = mem_df['used_percent'].mean()
        overall_max_mem = mem_df['used_percent'].max()
        overall_min_mem = mem_df['used_percent'].min()
        avg_used_gb = mem_df['used_gb'].mean()
        peak_used_gb = mem_df['used_gb'].max()
        print(f"- Memory utilization: avg={overall_avg_mem:.1f}%, peak={overall_max_mem:.1f}%, min={overall_min_mem:.1f}%")
        print(f"- Memory usage: avg={avg_used_gb:.1f}GB, peak={peak_used_gb:.1f}GB")
    else:
        print(f"- Memory utilization: ❌ Data not available")

    if not net_df.empty and 'bytes_recv_rate' in net_df.columns:
        avg_recv_rate = net_df['bytes_recv_rate'].mean() / (1024**2)
        avg_sent_rate = net_df['bytes_sent_rate'].mean() / (1024**2)
        avg_total_rate = avg_recv_rate + avg_sent_rate
        peak_recv_rate = net_df['bytes_recv_rate'].max() / (1024**2)
        peak_sent_rate = net_df['bytes_sent_rate'].max() / (1024**2)
        peak_total_rate = (net_df['bytes_recv_rate'] + net_df['bytes_sent_rate']).max() / (1024**2)
        print(f"- Network throughput: avg={avg_total_rate:.2f}MB/s, peak={peak_total_rate:.2f}MB/s")
        print(f"- Network breakdown: recv_avg={avg_recv_rate:.2f}MB/s, sent_avg={avg_sent_rate:.2f}MB/s")
    else:
        print(f"- Network throughput: ❌ Data not available")

else:
    print("❌ No tick data available for analysis")

In [ ]:
# ✅ FINAL TIME-FILTERED METRICS PREPARATION
print("✅ FINAL TIME-FILTERED METRICS PREPARATION")
print("=" * 50)

# Create properly named aligned datasets for plotting
if 'cpu_data_nodes' in locals():
    cpu_df_aligned = pd.concat(cpu_data_nodes, ignore_index=True) if cpu_data_nodes else pd.DataFrame()
else:
    cpu_df_aligned = pd.DataFrame()

if 'mem_data_nodes' in locals():
    mem_df_aligned = pd.concat(mem_data_nodes, ignore_index=True) if mem_data_nodes else pd.DataFrame()
else:
    mem_df_aligned = pd.DataFrame()

if 'net_data_nodes' in locals():
    net_df_aligned = pd.concat(net_data_nodes, ignore_index=True) if net_data_nodes else pd.DataFrame()
else:
    net_df_aligned = pd.DataFrame()

print(f"\n🎯 EXPERIMENT-ALIGNED SYSTEM METRICS:")
print("-" * 35)

if test_start_time is not None and test_end_time is not None:
    print(f"⏰ Test period identified: {test_duration_minutes:.2f} minutes")
    print(f"📅 Time range: {test_start_time:.1f} to {test_end_time:.1f}")
    print(f"🔧 Smart filtering applied to exclude pre-test startup data")
else:
    print(f"⚠️  Test period not identified - using all available data")
    print(f"📝 This may include Telegraf startup and post-test data")

print(f"\n📊 Final dataset sizes:")
print(f"   💻 CPU metrics: {len(cpu_df_aligned):,} records")
print(f"   💾 Memory metrics: {len(mem_df_aligned):,} records")  
print(f"   🌐 Network metrics: {len(net_df_aligned):,} records")

# Quick validation of time alignment
if not cpu_df_aligned.empty and 'timestamp_m' in cpu_df_aligned.columns:
    cpu_duration = cpu_df_aligned['timestamp_m'].max() - cpu_df_aligned['timestamp_m'].min()
    print(f"   ⏱️  CPU data spans: {cpu_duration:.1f} minutes")

if not mem_df_aligned.empty and 'timestamp_m' in mem_df_aligned.columns:
    mem_duration = mem_df_aligned['timestamp_m'].max() - mem_df_aligned['timestamp_m'].min()
    print(f"   ⏱️  Memory data spans: {mem_duration:.1f} minutes")

if not net_df_aligned.empty and 'timestamp_m' in net_df_aligned.columns:
    net_duration = net_df_aligned['timestamp_m'].max() - net_df_aligned['timestamp_m'].min()
    print(f"   ⏱️  Network data spans: {net_duration:.1f} minutes")

print(f"\n✅ Data preprocessing complete!")
print(f"🎨 Ready for time-aligned visualization and analysis")